# Are Dish Trends Real — or Just Noise? Temporal Evidence Mining in Yelp Reviews

**How do we detect genuine changes in cuisine popularity, dish sentiment, and review language over time?**

### The Core Objective
Task 5 adds a time dimension to the restaurant review project. Working with the same Yelp review corpus used in earlier tasks, you will slice reviews into time bins, track review volume, dish mention rates, lexicon sentiment, ratings, and distinctive vocabulary, and then distinguish real temporal patterns from sampling artifacts.

### Our Analytical Approach
1. **Temporal EDA:** Review volume and rating trends as baseline context.
2. **Dish mention trends:** Normalized mention rates by time bin — rising, stable, or declining?
3. **Lexicon sentiment over time:** Does sentiment about a dish track with rating changes?
4. **Temporal TF-IDF / term shift:** Which terms become more or less distinctive across periods?
5. **Notable-period inspection:** Return to real reviews to test whether the quantitative trends are meaningful.

### Why This Matters
Temporal patterns are among the most interpretation-sensitive results in the course. Without normalization, sparse bins, platform growth effects, and seasonal sampling can all create spurious trends. This task teaches you to ask "is this real?" before asking "what does it mean?"


## Table of Contents

1. [Runtime and Package Setup](#0.-Runtime-and-package-setup)
2. [Configuration](#1.-Configuration)
3. [Dataset Loading](#2.-Dataset-loading)
4. [Construct the Temporal Review Subset](#3.-Construct-the-temporal-review-subset)
5. [Temporal EDA](#4.-Temporal-EDA)
6. [Dish List](#5.-Load-or-construct-a-dish-list)
7. [Temporal Dish Mention Trends](#6.-Temporal-dish-mention-trends)
8. [Temporal Rating and Sentiment Trends](#7.-Temporal-rating-and-lightweight-sentiment-trends)
9. [Temporal Term Trends (TF-IDF)](#8.-Temporal-term-trends)
10. [Notable-Period Review Inspection](#10.-Notable-period-review-inspection)
11. [Required Comparison and Interpretation](#11.-Required-comparison-and-interpretation)
12. [Deliverables Checklist](#12.-Deliverables-checklist)


# Task 5 — Temporal Evidence Mining

Task 5 adds time to the Yelp review analysis. You will detect temporal patterns in dish
popularity, sentiment trends, and review language change. The analysis produces structured
temporal evidence tables that can support later tasks and direct qualitative interpretation.

## Pipeline Artifact Handoff

Task 5 produces temporal evidence tables for the capstone pipeline. Its key handoff artifacts
live under `artifacts/task5/`: `temporal_review_subset.csv`,
`review_counts_and_ratings_by_time.csv`, `dish_mentions_by_time_long.csv`,
`dish_sentiment_by_time.csv`, `temporal_sentiment_trends.csv`, and `top_terms_by_time.csv`.

## Required artifact contract for Task 5

Your completed notebook should produce these core artifacts under `artifacts/task5/`:

- `temporal_review_subset.csv` — the time-sliced review data.
- `time_axis_support.csv` — time bin support and data availability.
- `review_counts_and_ratings_by_time.csv` — volume and rating trends.
- `dish_mentions_by_time_long.csv` — dish popularity trends over time.
- `dish_sentiment_by_time.csv` — dish-level sentiment trends by time bin.
- `temporal_sentiment_trends.csv` — overall sentiment trends by time bin.
- `top_terms_by_time.csv` — distinctive terms in each time period.
- `task5_notable_period_annotations.csv` — notable periods identified.

## Learning objectives

By the end of this task, you should be able to:

- construct a time-sliced Yelp review subset with coherent time periods;
- perform basic temporal EDA: review volume and rating trends;
- identify temporal patterns in dish mentions, popularity, and sentiment;
- compute distinctive terms per time period using TF-IDF;
- interpret temporal changes as real patterns, seasonal effects, or sampling artifacts.

## 0. Runtime and package setup

### Local installation and run

You can download and run this notebook on your personal computer. Running locally is recommended for this task — temporal analysis over large review files benefits from faster I/O, .

[Jupyter Installation Instructions](https://docs.jupyter.org/en/latest/install.html)

### Google Colab

You can also run this notebook on Google Colab.

- Use **Runtime → Change runtime type → Python 3**.
- A GPU is not required for the required path.

The notebook is designed to run in:

- `smoke_test` mode with no network access;
- `manual` mode using local Yelp JSON files;
- `kaggle_latest` mode when KaggleHub and credentials/network access are available.


## *Smoke Test Mode*

*Use `smoke_test` mode only to verify that your notebook runs end-to-end when real data is unavailable. The synthetic fixture has deliberately planted temporal signals (rising, declining, stable dishes), but actual trend directions are predetermined. Base your final report conclusions and interpretation on the real Yelp dataset.*


In [ ]:
# Optional package installation for Colab.
# Keep this False for local smoke tests or controlled course environments.
INSTALL_OPTIONAL_PACKAGES = False

if INSTALL_OPTIONAL_PACKAGES:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])


In [ ]:
import os
import re
import json
import math
import zipfile
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

print("Imports complete.")

In [ ]:
# Xiaofei's code — Task 5 local path bootstrap / 本地真实数据与输出路径配置
# 中文注释：这个 cell 要在 Section 1 Configuration 之前运行，用来避免最终分析误用 smoke_test。

# 中文注释：重新导入 os，确保单独运行本 cell 时也能设置环境变量。
import os
# 中文注释：重新导入 Path，确保单独运行本 cell 时也能处理路径。
from pathlib import Path

# 中文注释：定义项目根目录查找函数，避免从 notebooks 子目录运行时输出到错误位置。
def find_project_root(start: Path) -> Path:
    # 中文注释：从当前目录开始逐级向上检查候选目录。
    for candidate in [start, *start.parents]:
        # 中文注释：项目根目录应包含 notebooks 目录。
        has_notebooks = (candidate / "notebooks").is_dir()
        # 中文注释：最终分析必须使用真实 Yelp 数据目录。
        has_raw_data = (candidate / "data" / "raw").is_dir()
        # 中文注释：Task 5 需要读取 Task 3 生成的最终菜品表。
        has_task3_artifact = (candidate / "artifacts" / "task3" / "final_dish_list.csv").is_file()
        # 中文注释：Task 5 优先复用 Task 4 的目标商家选择结果。
        has_task4_artifact = (candidate / "artifacts" / "task4" / "target_business_selection.csv").is_file()
        # 中文注释：本项目通常包含本地虚拟环境或项目包目录。
        has_project_package = (candidate / "cs598capstone").is_dir()
        # 中文注释：同时具备 notebooks 且具备真实数据、上游输出或项目包时，认为是项目根目录。
        if has_notebooks and (has_raw_data or has_task3_artifact or has_task4_artifact or has_project_package):
            # 中文注释：返回识别出的项目根目录。
            return candidate
    # 中文注释：如果没有找到严格匹配的根目录，则退回到启动目录。
    return start

# 中文注释：记录前面初始化 cell 可能已经创建的旧输出目录。
bootstrap_artifact_dir = globals().get("ARTIFACT_DIR")
# 中文注释：优先读取显式配置的项目根目录。
env_root = os.environ.get("DMC_PROJECT_ROOT", "").strip()
# 中文注释：如果用户配置了项目根目录，则使用该路径。
if env_root:
    # 中文注释：展开并规范化用户提供的项目根目录。
    PROJECT_ROOT = Path(env_root).expanduser().resolve()
# 中文注释：否则从当前工作目录自动向上查找项目根目录。
else:
    # 中文注释：根据当前目录自动识别项目根目录。
    PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# 中文注释：切换到项目根目录，保证所有相对路径都写入根目录下的 artifacts。
os.chdir(PROJECT_ROOT)

# 中文注释：强制使用真实本地 Yelp 数据，不使用 smoke_test。
os.environ["DMC_DATA_SOURCE"] = "manual"
# 中文注释：同步设置 Task 5 专用数据来源变量，避免旧环境变量回落到 smoke_test。
os.environ["TASK5_DATA_SOURCE"] = "manual"
# 中文注释：指定真实 Yelp JSON 数据目录。
os.environ["DMC_DATA_DIR"] = str(PROJECT_ROOT / "data" / "raw")
# 中文注释：同步设置 Task 5 专用数据目录变量。
os.environ["TASK5_DATA_DIR"] = str(PROJECT_ROOT / "data" / "raw")
# 中文注释：指定 Task 3 传给 Task 5 的最终菜品表路径。
os.environ["TASK5_DISH_LIST_PATH"] = str(PROJECT_ROOT / "artifacts" / "task3" / "final_dish_list.csv")
# 中文注释：指定 Task 4 输出目录，供 Task 5 复用覆盖率和商家选择结果。
os.environ["TASK5_TASK4_ARTIFACT_DIR"] = str(PROJECT_ROOT / "artifacts" / "task4")
# 中文注释：指定 Task 4 的目标商家选择表路径。
os.environ["TASK5_TASK4_TARGET_BUSINESS_SELECTION_PATH"] = str(PROJECT_ROOT / "artifacts" / "task4" / "target_business_selection.csv")
# 中文注释：指定 Task 4 的菜品覆盖率表路径。
os.environ["TASK5_TASK4_COVERAGE_PATH"] = str(PROJECT_ROOT / "artifacts" / "task4" / "dish_mention_coverage.csv")
# 中文注释：读取 Task 4 目标商家选择表路径，存在时优先复用以保持 Task 4 和 Task 5 的分析对象一致。
task4_target_business_path = Path(os.environ["TASK5_TASK4_TARGET_BUSINESS_SELECTION_PATH"])
# 中文注释：如果 Task 4 商家选择表存在，则启用 Task 4 handoff。
if task4_target_business_path.exists():
    # 中文注释：启用 Task 4 目标商家选择。
    os.environ["TASK5_USE_TASK4_TARGET_BUSINESSES"] = "1"
# 中文注释：如果 Task 4 商家选择表不存在，则回退到真实 Yelp 数据的类别扫描。
else:
    # 中文注释：关闭 Task 4 目标商家选择，后续 loader 会使用 raw_category_scan。
    os.environ["TASK5_USE_TASK4_TARGET_BUSINESSES"] = "0"
# 中文注释：指定 Task 5 的所有输出 artifact 目录。
TASK5_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "task5"
# 中文注释：把输出目录写入 Task 5 专用环境变量。
os.environ["TASK5_ARTIFACT_DIR"] = str(TASK5_OUTPUT_DIR)
# 中文注释：重设 notebook 全局输出目录，覆盖可能来自旧相对路径的输出目录。
ARTIFACT_DIR = TASK5_OUTPUT_DIR
# 中文注释：确保 Task 5 输出目录存在。
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 中文注释：定义本地句向量模型目录，供 embedding drift 分析优先离线使用。
LOCAL_EMBEDDING_MODEL_DIR = PROJECT_ROOT / "models" / "all-MiniLM-L6-v2"
# 中文注释：如果本地句向量模型目录存在，则使用本地模型避免运行时下载。
if LOCAL_EMBEDDING_MODEL_DIR.exists():
    # 中文注释：把本地 sentence-transformers 模型路径写入 Task 5 配置。
    os.environ["TASK5_EMBEDDING_MODEL_PATH"] = str(LOCAL_EMBEDDING_MODEL_DIR)
    # 中文注释：本地模型存在时关闭下载，保证结果可复现。
    os.environ["TASK5_ALLOW_EMBEDDING_DOWNLOAD"] = "0"
# 中文注释：如果本地模型不存在，则后续代码会自动使用 TF-IDF+SVD fallback。
else:
    # 中文注释：清空本地模型路径，避免误用不存在的目录。
    os.environ["TASK5_EMBEDDING_MODEL_PATH"] = ""
    # 中文注释：关闭下载并让 notebook 使用可复现的本地 fallback。
    os.environ["TASK5_ALLOW_EMBEDDING_DOWNLOAD"] = "0"

# 中文注释：如果前面 cell 创建了错误位置的空输出目录，则安全清理空目录。
if bootstrap_artifact_dir is not None:
    # 中文注释：把旧输出目录转换为 Path 对象。
    bootstrap_artifact_dir = Path(bootstrap_artifact_dir)
    # 中文注释：只有旧目录和真实输出目录不一致时才尝试清理。
    if bootstrap_artifact_dir.resolve() != ARTIFACT_DIR.resolve():
        # 中文注释：尝试删除错误位置下的空 task5 目录。
        try:
            # 中文注释：只删除空目录，目录内有文件时会自动保留。
            bootstrap_artifact_dir.rmdir()
        # 中文注释：目录不存在或非空时不影响 notebook 继续运行。
        except OSError:
            # 中文注释：跳过无法安全删除的旧目录。
            pass
        # 中文注释：尝试删除错误位置下的空 artifacts 父目录。
        try:
            # 中文注释：只删除空父目录，避免误删已有输出。
            bootstrap_artifact_dir.parent.rmdir()
        # 中文注释：父目录不存在或非空时不影响 notebook 继续运行。
        except OSError:
            # 中文注释：跳过无法安全删除的旧父目录。
            pass

# 中文注释：列出最终分析必须存在的真实 Yelp 数据文件。
required_yelp_files = [PROJECT_ROOT / "data" / "raw" / "yelp_academic_dataset_business.json", PROJECT_ROOT / "data" / "raw" / "yelp_academic_dataset_review.json"]
# 中文注释：检查真实 Yelp 数据文件是否缺失。
missing_yelp_files = [path for path in required_yelp_files if not path.exists()]
# 中文注释：如果缺少真实数据文件，则立即报错，避免回退到 smoke_test。
if missing_yelp_files:
    # 中文注释：抛出包含缺失文件路径的错误。
    raise FileNotFoundError(f"Task 5 requires real Yelp data files, missing: {missing_yelp_files}")
# 中文注释：如果数据来源仍然是 smoke_test，则立即报错。
if os.environ.get("DMC_DATA_SOURCE") == "smoke_test" or os.environ.get("TASK5_DATA_SOURCE") == "smoke_test":
    # 中文注释：禁止最终分析使用 smoke_test。
    raise ValueError("Task 5 final analysis must use manual data, not smoke_test.")
# 中文注释：检查 Task 3 菜品表是否存在。
if not Path(os.environ["TASK5_DISH_LIST_PATH"]).exists():
    # 中文注释：缺少 Task 3 菜品表时立即报错，避免使用教学 fallback 菜品表。
    raise FileNotFoundError(f"Task 5 requires Task 3 dish list: {os.environ['TASK5_DISH_LIST_PATH']}")
# 中文注释：检查 Task 4 目标商家选择表是否存在；缺失时不报错，因为 Task 5 可用真实 Yelp 类别扫描回退。
if not Path(os.environ["TASK5_TASK4_TARGET_BUSINESS_SELECTION_PATH"]).exists():
    # 中文注释：打印回退提示，避免用户误以为仍在使用 Task 4 handoff。
    print("Task 4 target business selection not found; Task 5 will use raw Yelp category scan.")

# 中文注释：打印项目根目录，便于检查路径是否正确。
print("PROJECT_ROOT:", PROJECT_ROOT)
# 中文注释：打印当前工作目录，确认已经切换到项目根目录。
print("cwd:", os.getcwd())
# 中文注释：打印真实数据来源配置。
print("DATA_SOURCE will be:", os.environ.get("DMC_DATA_SOURCE"))
# 中文注释：打印真实数据目录配置。
print("DMC_DATA_DIR will be:", os.environ.get("DMC_DATA_DIR"))
# 中文注释：打印 Task 3 菜品表路径配置。
print("TASK5_DISH_LIST_PATH:", os.environ.get("TASK5_DISH_LIST_PATH"))
# 中文注释：打印 Task 4 商家选择路径配置。
print("TASK5_TASK4_TARGET_BUSINESS_SELECTION_PATH:", os.environ.get("TASK5_TASK4_TARGET_BUSINESS_SELECTION_PATH"))
# 中文注释：打印本地 embedding 模型路径配置。
print("TASK5_EMBEDDING_MODEL_PATH:", os.environ.get("TASK5_EMBEDDING_MODEL_PATH") or "tfidf_svd_fallback")
# 中文注释：打印 Task 5 输出目录配置。
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())


## 1. Configuration

The setup worked solution defaults to `smoke_test` mode so the notebook can be executed and validated without network access.

For real data, set:

```python
DATASET_MODE = "manual"
DATA_ROOT = Path("/path/to/yelp_json_files")
```

or use `kaggle_latest` if KaggleHub is configured.

## About Temporal Aggregation / Rolling Trend Analysis

Temporal aggregation groups reviews, dish mentions, ratings, and sentiment into month, quarter, or year bins so students can reason about change over time. In Task 5, rolling summaries smooths noisy time series and highlights patterns that may be worth interpreting, while minimum-bin thresholds prevent overreading tiny samples.

This method is the backbone of temporal data mining in the notebook. It teaches students to distinguish raw volume from normalized rates, supported bins from sparse bins, and descriptive trends from forecasts or causal claims. Rolling summaries can make patterns easier to see, but they can also hide abrupt changes, so you should pair plots with support counts and example reviews.

**Key Resources:**
- [pandas time series documentation](https://pandas.pydata.org/docs/user_guide/timeseries.html) - Date indexing and time aggregation tools.
- [pandas rolling windows](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html) - Rolling aggregation API.
- [Wikipedia: Moving average](https://en.wikipedia.org/wiki/Moving_average) - Conceptual background for smoothing.

In [ ]:
DATASET_MODE = os.environ.get("TASK5_DATA_SOURCE", os.environ.get("DMC_DATA_SOURCE", "smoke_test"))  # smoke_test, manual, kaggle_latest

# Used only in manual mode.
DATA_ROOT = Path(os.environ.get("TASK5_DATA_DIR", os.environ.get("DMC_DATA_DIR", "data/yelp")))

# Used only in kaggle_latest mode.
KAGGLE_DATASET = os.environ.get("TASK5_KAGGLE_DATASET", "yelp-dataset/yelp-dataset")

TARGET_CATEGORY = os.environ.get("TASK5_TARGET_CATEGORY", "Chinese")
REQUIRE_RESTAURANTS = os.environ.get("TASK5_REQUIRE_RESTAURANTS", "1").lower() in {"1", "true", "yes", "y"}

TIME_GRANULARITY = os.environ.get("TASK5_TIME_GRANULARITY", "quarter")  # month, quarter, year
MIN_REVIEWS_PER_TIME_BIN = int(os.environ.get("TASK5_MIN_REVIEWS_PER_TIME_BIN", "20"))

MAX_BUSINESSES = int(os.environ.get("TASK5_MAX_BUSINESSES", "80"))
MAX_REVIEWS_TOTAL = int(os.environ.get("TASK5_MAX_REVIEWS_TOTAL", "10000"))
MAX_REVIEWS_PER_BUSINESS = int(os.environ.get("TASK5_MAX_REVIEWS_PER_BUSINESS", "120"))
MAX_DISHES = int(os.environ.get("TASK5_MAX_DISHES", "12"))
MAX_TERMS_FOR_HEATMAP = int(os.environ.get("TASK5_MAX_TERMS_FOR_HEATMAP", "24"))
MAX_EXAMPLE_REVIEWS = int(os.environ.get("TASK5_MAX_EXAMPLE_REVIEWS", "8"))
MIN_DISH_MENTION_REVIEWS_PER_BIN = int(os.environ.get("TASK5_MIN_DISH_MENTION_REVIEWS_PER_BIN", "2"))

ROLLING_WINDOW = int(os.environ.get("TASK5_ROLLING_WINDOW", "2"))  # used for optional smoothing in plots

print("Task 5 core config loaded:")
print({
    "DATASET_MODE": DATASET_MODE,
    "TARGET_CATEGORY": TARGET_CATEGORY,
    "TIME_GRANULARITY": TIME_GRANULARITY,
    "MAX_BUSINESSES": MAX_BUSINESSES,
    "MAX_REVIEWS_TOTAL": MAX_REVIEWS_TOTAL,
})

### What your configuration choices mean

Every constant set in this cell has downstream consequences you should understand before running the full notebook.

**`TIME_GRANULARITY`** — `quarter` is the recommended default for Yelp data because monthly bins are often too sparse (< 20 reviews) for most businesses, and yearly bins smooth over real intra-year variation. If you switch to `month`, inspect `time_axis_support.csv` carefully: many bins will be flagged `sparse_below_threshold` and dropped.

**`MIN_REVIEWS_PER_TIME_BIN`** — this threshold gates every downstream analysis. A bin with 5 reviews produces unreliable means; a bin with 20+ reviews gives you something defensible. Raising this threshold improves claim quality but reduces temporal resolution.

**`MAX_BUSINESSES` and `MAX_REVIEWS_TOTAL`** — both caps must be consistent. If you raise `MAX_BUSINESSES` to 200 but keep `MAX_REVIEWS_TOTAL` at 10,000, each business gets roughly 50 reviews on average — not enough for stable per-business trends.

**`MAX_DISHES`** — the dish list from Task 3 may contain many items, but tracking too many dishes at once makes the heatmaps unreadable and the analysis unfocused. Twelve dishes is a reasonable upper bound; for your report, you will probably focus on three to five.

> **Student note:** Changing these constants after running the notebook means rerunning from the data-loading cell forward. If you change `TIME_GRANULARITY`, you must rerun *everything* because time bins affect the temporal subset, the dish mention counts, the sentiment trends, the term trends, and the evidence-grounded corpus.

## About TF-IDF Vectorization

TF-IDF vectorization converts text into weighted term features that increase the importance of terms that are common within a time bin but relatively uncommon across the full temporal corpus. In Task 5, it powers the temporal term-shift analysis: identifying which vocabulary is distinctive in each period.

Because TF-IDF remains sparse and inspectable, it supports direct interpretation. You can explain which terms became distinctive in a given period by examining the top-weighted features directly, then checking example reviews before making claims.

**Key Resources:**
- [scikit-learn TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) - TF-IDF implementation reference.
- [scikit-learn Feature Extraction](https://scikit-learn.org/stable/modules/feature_extraction.html) - Text feature extraction guide.


In [ ]:
def save_table(df, name, index=False):
    path = ARTIFACT_DIR / name
    df.to_csv(path, index=index)
    print(f"Saved {path} ({len(df):,} rows)")
    return path

def save_figure(name):
    path = ARTIFACT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print(f"Saved {path}")
    return path

def normalize_categories(categories):
    if categories is None or (isinstance(categories, float) and pd.isna(categories)):
        return []
    if isinstance(categories, list):
        return [str(c).strip() for c in categories if str(c).strip()]
    return [c.strip() for c in str(categories).split(",") if c.strip()]

def category_matches(categories, target):
    cats = normalize_categories(categories)
    target_lower = target.lower()
    return any(c.lower() == target_lower for c in cats)

def is_restaurant_or_food(categories):
    cats = {c.lower() for c in normalize_categories(categories)}
    if not cats:
        return False
    parent_ok = {"restaurants", "food"} & cats
    not_ok = {"shopping", "nightlife", "event planning & services"}
    return bool(parent_ok) and not bool(cats <= not_ok)



### Utility helpers: time bins are the backbone of this analysis

The next code block defines the time-binning utilities that every downstream section depends on. A few things worth understanding:

**`time_bin_from_date`** converts a datetime to a period string such as `2018Q3`. Pandas period strings sort lexicographically *and* chronologically for quarter and year formats, which is why `natural_sort_time_bins` can fall back to `pd.Period(x).start_time` for correct ordering.

**`complete_time_bin_range`** is a key design decision: it fills in *every* quarter between the first and last observed quarters, including quarters with zero reviews. This means trend plots always show a continuous x-axis with visible gaps, rather than silently skipping quarters where data is sparse. A sparse quarter looks very different from a quarter with no reviews at all — both appear on the axis, but only `supported` bins feed into quantitative estimates.

> **Pay attention:** if your x-axis has large gaps (many gray or yellow bins in a row), your temporal claims are fragile. Long gap sequences usually mean the business set changed, the dataset has a coverage hole, or your `MIN_REVIEWS_PER_TIME_BIN` is too high for the available data.

In [ ]:
def find_file(root, patterns):
    root = Path(root)
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None

def extract_nested_zips(root):
    root = Path(root)
    for zip_path in root.rglob("*.zip"):
        extract_dir = zip_path.with_suffix("")
        marker = extract_dir / ".extracted"
        if marker.exists():
            continue
        extract_dir.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)
            marker.write_text("ok", encoding="utf-8")
            print("Extracted", zip_path, "to", extract_dir)
        except Exception as exc:
            print("Could not extract", zip_path, ":", exc)

def time_bin_from_date(series, granularity):
    dates = pd.to_datetime(series, errors="coerce")
    if granularity == "month":
        return dates.dt.to_period("M").astype(str)
    if granularity == "quarter":
        return dates.dt.to_period("Q").astype(str)
    if granularity == "year":
        return dates.dt.to_period("Y").astype(str)
    raise ValueError("TIME_GRANULARITY must be one of: month, quarter, year")

def natural_sort_time_bins(values):
    return sorted(values, key=lambda x: pd.Period(x).start_time if x and x != "NaT" else pd.Timestamp.min)


def complete_time_bin_range(values, granularity):
    """Return every time bin between the first and last observed bin.

    Trend plots use this full axis so sparse or absent periods show up as visible
    gaps instead of disappearing from the x-axis after filtering.
    """
    freq_map = {"month": "M", "quarter": "Q", "year": "Y"}
    freq = freq_map.get(granularity)
    if freq is None:
        raise ValueError(f"Unsupported TIME_GRANULARITY: {granularity}")
    clean_values = [str(v) for v in values if str(v) and str(v) not in {"nan", "NaT"}]
    if not clean_values:
        return []
    periods = pd.PeriodIndex(clean_values, freq=freq)
    return [str(p) for p in pd.period_range(periods.min(), periods.max(), freq=freq)]

print("Helpers ready.")

## 2. Dataset loading

For the real Yelp dataset, the notebook streams review JSON lines and only keeps reviews for the selected business IDs. This avoids loading the full review file into memory.

When available, Task 5 now reuses Task 4's `target_business_selection.csv` rather than taking the first matching businesses from the raw Yelp business file. That keeps the temporal analysis aligned with the same focused Chinese restaurant sample used for dish-local ranking and avoids accidental category term shift from arbitrary file order.

In [ ]:
def make_smoke_test_yelp_data():
    """Create a small Yelp-shaped dataset with intentional temporal signals."""
    businesses = []
    categories = [
        ("b_ch_1", "Golden Dragon", ["Restaurants", "Chinese", "Szechuan"]),
        ("b_ch_2", "Dumpling House", ["Restaurants", "Chinese", "Dim Sum"]),
        ("b_ch_3", "Noodle Garden", ["Restaurants", "Chinese", "Noodles"]),
        ("b_ch_4", "Hot Pot City", ["Restaurants", "Chinese", "Hot Pot"]),
        ("b_ch_5", "Mapo Kitchen", ["Restaurants", "Chinese", "Szechuan"]),
        ("b_mx_1", "Taqueria Sol", ["Restaurants", "Mexican", "Tacos"]),
        ("b_th_1", "Thai Basil", ["Restaurants", "Thai"]),
    ]
    for bid, name, cats in categories:
        businesses.append({
            "business_id": bid,
            "name": name,
            "categories": ", ".join(cats),
            "city": "Smokeville",
            "state": "CA",
            "stars": round(3.5 + random.random(), 1),
            "review_count": 100,
        })

    positive = ["excellent", "great", "fresh", "crispy", "rich", "favorite", "delicious", "friendly", "perfect"]
    negative = ["bland", "slow", "cold", "greasy", "disappointing", "salty", "overpriced", "bad"]
    dishes = ["dumplings", "noodles", "fried rice", "hot pot", "mapo tofu", "kung pao chicken", "spring rolls"]

    reviews = []
    rid = 0
    quarters = pd.period_range("2019Q1", "2023Q4", freq="Q")
    for q_i, period in enumerate(quarters):
        q_start = period.start_time
        # Dumplings rise over time; hot pot has winter seasonality; fried rice stays steady.
        dumpling_weight = 1 + q_i * 0.12
        hotpot_weight = 2.0 if period.quarter in [1, 4] else 0.6
        dish_weights = {
            "dumplings": dumpling_weight,
            "noodles": 1.2,
            "fried rice": 1.0,
            "hot pot": hotpot_weight,
            "mapo tofu": 0.9 + (0.5 if q_i > 10 else 0),
            "kung pao chicken": 0.8,
            "spring rolls": 0.5,
        }
        dish_names = list(dish_weights)
        probs = np.array([dish_weights[d] for d in dish_names], dtype=float)
        probs = probs / probs.sum()

        n_reviews = 22 + (q_i % 4) * 3
        for _ in range(n_reviews):
            bid = random.choice(["b_ch_1", "b_ch_2", "b_ch_3", "b_ch_4", "b_ch_5"])
            dish = np.random.choice(dish_names, p=probs)
            # Later periods become slightly more positive for dumplings/mapo; service complaints increase late.
            base_star = 3.4 + 0.02 * q_i
            if dish in ["dumplings", "mapo tofu"] and q_i > 8:
                base_star += 0.5
            if dish == "hot pot" and period.quarter in [1, 4]:
                base_star += 0.3
            service_complaint = q_i > 14 and random.random() < 0.25
            if service_complaint:
                base_star -= 0.8
            stars = int(np.clip(round(base_star + np.random.normal(0, 0.8)), 1, 5))
            pos_word = random.choice(positive)
            neg_word = random.choice(negative)
            if stars >= 4:
                text = f"The {dish} was {pos_word} and the flavors were memorable. Service was friendly and I would order it again."
            elif stars <= 2:
                text = f"The {dish} was {neg_word}. I wanted to like this place but the visit felt disappointing."
            else:
                text = f"The {dish} was okay. Some bites were good but the experience was mixed."
            if service_complaint:
                text += " The service was slow even though the food had potential."
            reviews.append({
                "review_id": f"r_{rid}",
                "business_id": bid,
                "stars": stars,
                "date": (q_start + pd.Timedelta(days=random.randint(0, 80))).strftime("%Y-%m-%d"),
                "text": text,
            })
            rid += 1

    # Add a few non-target reviews to test filtering.
    for i in range(30):
        bid = random.choice(["b_mx_1", "b_th_1"])
        reviews.append({
            "review_id": f"r_other_{i}",
            "business_id": bid,
            "stars": random.randint(2, 5),
            "date": (pd.Timestamp("2021-01-01") + pd.Timedelta(days=random.randint(0, 700))).strftime("%Y-%m-%d"),
            "text": "The tacos or curry were tasty but this is not part of the Chinese subset.",
        })
    return pd.DataFrame(businesses), pd.DataFrame(reviews)

def resolve_dataset_root():
    if DATASET_MODE == "manual":
        return Path(DATA_ROOT)
    if DATASET_MODE == "kaggle_latest":
        try:
            import kagglehub
            path = kagglehub.dataset_download(KAGGLE_DATASET)
            return Path(path)
        except Exception as exc:
            raise RuntimeError(
                "Could not download Kaggle dataset. Use DATASET_MODE='manual' with local files, "
                "or DATASET_MODE='smoke_test'. Original error: " + repr(exc)
            )
    raise ValueError(f"Unsupported DATASET_MODE for real loading: {DATASET_MODE}")


def load_task4_target_business_selection() -> pd.DataFrame:
    """Load the focused Task 4 business sample when available."""
    if not USE_TASK4_TARGET_BUSINESS_SELECTION or not TASK4_TARGET_BUSINESS_SELECTION_PATH.exists():
        return pd.DataFrame()
    selection = pd.read_csv(TASK4_TARGET_BUSINESS_SELECTION_PATH)
    if "business_id" not in selection.columns:
        warnings.warn(f"{TASK4_TARGET_BUSINESS_SELECTION_PATH} has no business_id column; falling back to raw category scan.")
        return pd.DataFrame()
    selection = selection.dropna(subset=["business_id"]).drop_duplicates("business_id").copy()
    if selection.empty:
        return pd.DataFrame()
    rename = {
        "business_stars": "stars",
        "business_review_count": "review_count",
    }
    selection = selection.rename(columns={k: v for k, v in rename.items() if k in selection.columns})
    for col in ["name", "categories", "city", "state", "stars", "review_count"]:
        if col not in selection.columns:
            selection[col] = np.nan
    selection = selection.head(MAX_BUSINESSES).copy()
    selection["task5_business_selection_source"] = "task4_target_business_selection"
    return selection


def load_yelp_real_subset():
    root = resolve_dataset_root()
    extract_nested_zips(root)

    business_file = find_file(root, ["*business*.json", "yelp_academic_dataset_business.json"])
    review_file = find_file(root, ["*review*.json", "yelp_academic_dataset_review.json"])

    if business_file is None or review_file is None:
        raise FileNotFoundError(
            f"Could not find Yelp business/review JSON files under {root}. "
            "Expected files like yelp_academic_dataset_business.json and yelp_academic_dataset_review.json."
        )

    task4_selection = load_task4_target_business_selection()
    if not task4_selection.empty:
        business_df = task4_selection.copy()
    else:
        businesses = []
        with open(business_file, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                cats = obj.get("categories")
                if REQUIRE_RESTAURANTS and not is_restaurant_or_food(cats):
                    continue
                if not category_matches(cats, TARGET_CATEGORY):
                    continue
                businesses.append({
                    "business_id": obj.get("business_id"),
                    "name": obj.get("name"),
                    "categories": obj.get("categories"),
                    "city": obj.get("city"),
                    "state": obj.get("state"),
                    "stars": obj.get("stars"),
                    "review_count": obj.get("review_count"),
                    "task5_business_selection_source": "raw_category_scan",
                })
                if len(businesses) >= MAX_BUSINESSES:
                    break
        business_df = pd.DataFrame(businesses).dropna(subset=["business_id"])

    if business_df.empty:
        raise ValueError(f"No businesses found for TARGET_CATEGORY={TARGET_CATEGORY!r}.")

    target_business_ids = set(business_df["business_id"].tolist())
    per_business_counts = Counter()
    reviews = []

    with open(review_file, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            bid = obj.get("business_id")
            if bid not in target_business_ids:
                continue
            if per_business_counts[bid] >= MAX_REVIEWS_PER_BUSINESS:
                continue
            text = obj.get("text")
            date = obj.get("date")
            if not text or not date:
                continue
            reviews.append({
                "review_id": obj.get("review_id"),
                "business_id": bid,
                "stars": obj.get("stars"),
                "date": date,
                "text": text,
            })
            per_business_counts[bid] += 1
            if len(reviews) >= MAX_REVIEWS_TOTAL:
                break

    review_df = pd.DataFrame(reviews)
    if review_df.empty:
        raise ValueError("No reviews found for selected businesses.")
    return business_df, review_df



### Understanding the smoke dataset

The `make_smoke_test_yelp_data()` function above creates a small synthetic Yelp-shaped dataset with *intentional* temporal patterns baked in:

- Some dishes show a rising trend over time (more mentions in recent quarters).
- Some businesses show improving or declining average ratings over time.
- Review volume varies across quarters to mimic real Yelp coverage patterns.

This is important: the smoke data is designed so that the methods you are about to implement *should detect these patterns*. If your smoke run shows flat trends for every dish and uniform ratings for every quarter, something is wrong upstream — likely in the time-bin construction or the dish matching step.

**In real Kaggle mode**, these patterns emerge organically from thousands of reviews. Some patterns will be clear; others will be noisy or ambiguous. That ambiguity is the point. A good report discusses whether a pattern is genuine or an artifact of sampling, sparsity, or platform behavior.

In [ ]:
TASK4_ARTIFACT_DIR = Path(os.environ.get("TASK5_TASK4_ARTIFACT_DIR", "artifacts/task4"))
TASK4_TARGET_BUSINESS_SELECTION_PATH = Path(
    os.environ.get("TASK5_TASK4_TARGET_BUSINESS_SELECTION_PATH", TASK4_ARTIFACT_DIR / "target_business_selection.csv")
)
USE_TASK4_TARGET_BUSINESS_SELECTION = os.environ.get("TASK5_USE_TASK4_TARGET_BUSINESSES", "1").lower() in {"1", "true", "yes", "y"}

ARTIFACT_DIR = Path(os.environ.get("TASK5_ARTIFACT_DIR", os.environ.get("DMC_ARTIFACT_DIR", "artifacts/task5")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset mode:", DATASET_MODE)
print("Target category:", TARGET_CATEGORY)
print("Artifact directory:", ARTIFACT_DIR.resolve())

if DATASET_MODE == "smoke_test":
    business_df, review_df = make_smoke_test_yelp_data()
    business_df["task5_business_selection_source"] = "smoke_test_fixture"
else:
    business_df, review_df = load_yelp_real_subset()

save_table(business_df, "task5_target_business_selection_used.csv")
sampling_diagnostics = pd.DataFrame([
    {"metric": "dataset_mode", "value": DATASET_MODE},
    {"metric": "target_category", "value": TARGET_CATEGORY},
    {"metric": "business_selection_source", "value": business_df.get("task5_business_selection_source", pd.Series(["unknown"])).iloc[0]},
    {"metric": "task4_target_business_selection_path", "value": str(TASK4_TARGET_BUSINESS_SELECTION_PATH)},
    {"metric": "selected_businesses", "value": int(business_df["business_id"].nunique())},
    {"metric": "reviews_collected", "value": int(len(review_df))},
    {"metric": "max_reviews_total", "value": int(MAX_REVIEWS_TOTAL)},
    {"metric": "max_reviews_per_business", "value": int(MAX_REVIEWS_PER_BUSINESS)},
])
if "competing_cuisine_tag_count" in business_df.columns:
    sampling_diagnostics = pd.concat([
        sampling_diagnostics,
        pd.DataFrame([
            {"metric": "businesses_with_zero_competing_cuisine_tags", "value": int((pd.to_numeric(business_df["competing_cuisine_tag_count"], errors="coerce").fillna(0) == 0).sum())},
            {"metric": "businesses_with_any_competing_cuisine_tags", "value": int((pd.to_numeric(business_df["competing_cuisine_tag_count"], errors="coerce").fillna(0) > 0).sum())},
        ]),
    ], ignore_index=True)
save_table(sampling_diagnostics, "task5_sampling_diagnostics.csv")

print("Businesses:", business_df.shape)
print("Reviews:", review_df.shape)
print("Business selection source:", business_df.get("task5_business_selection_source", pd.Series(["unknown"])).iloc[0])
display(business_df.head())
display(review_df.head())

## 3. Construct the temporal review subset

We now join reviews to business metadata, parse dates, assign time bins, and drop under-supported time bins.

The minimum time-bin threshold is important: a visual trend based on a tiny number of reviews is usually not reliable.

The notebook saves both the full time-bin support table and the dropped sparse bins so students can see what evidence was excluded before interpretation.

In [ ]:
review_df["date"] = pd.to_datetime(review_df["date"], errors="coerce")
review_df = review_df.dropna(subset=["date", "text", "stars", "business_id"]).copy()

subset_df = review_df.merge(
    business_df[["business_id", "name", "categories", "city", "state"]],
    on="business_id",
    how="left",
)

subset_df["time_bin"] = time_bin_from_date(subset_df["date"], TIME_GRANULARITY)
subset_df = subset_df[subset_df["time_bin"] != "NaT"].copy()
subset_df["stars"] = pd.to_numeric(subset_df["stars"], errors="coerce")
subset_df = subset_df.dropna(subset=["stars"]).copy()
subset_df["stars"] = subset_df["stars"].astype(float)



### Building the time axis: supported vs. unsupported bins

The code below constructs the *full* time axis — from the earliest to the latest quarter in the dataset — and labels each bin with one of three statuses:

| Status | Meaning | Used in analysis? |
|---|---|---|
| `supported` | ≥ `MIN_REVIEWS_PER_TIME_BIN` reviews | ✅ Yes |
| `sparse_below_threshold` | Reviews present but below threshold | ⚠️ Shown but flagged |
| `missing_from_raw_subset` | No reviews in this period for this category | ❌ No |

> **Why this matters:** if you simply dropped sparse bins from the x-axis, a plot covering 2010–2022 might show 2014–2016 immediately adjacent to 2018–2020, hiding a four-year gap. The full axis makes gaps visible. Yellow shading in plots = sparse bin; gray = missing bin. Never interpret a trend across a gap without discussing why the gap exists.

`time_axis_support.csv` is the source-of-truth table for all downstream sparsity discussions. You will need to reference it in your report.

In [ ]:
observed_time_order = natural_sort_time_bins(subset_df["time_bin"].unique())
complete_time_order = complete_time_bin_range(observed_time_order, TIME_GRANULARITY)
time_counts = subset_df["time_bin"].value_counts()

time_axis_support = pd.DataFrame({"time_bin": complete_time_order})
time_axis_support["review_count"] = time_axis_support["time_bin"].map(time_counts).fillna(0).astype(int)
time_axis_support["observed_in_raw_subset"] = time_axis_support["review_count"] > 0
time_axis_support["meets_min_reviews"] = time_axis_support["review_count"] >= MIN_REVIEWS_PER_TIME_BIN
time_axis_support["time_axis_status"] = np.select(
    [
        ~time_axis_support["observed_in_raw_subset"],
        time_axis_support["observed_in_raw_subset"] & ~time_axis_support["meets_min_reviews"],
    ],
    ["missing_from_raw_subset", "sparse_below_threshold"],
    default="supported",
)
time_axis_support["dropped_before_analysis"] = ~time_axis_support["meets_min_reviews"]
save_table(time_axis_support, "time_bin_support_all.csv")
save_table(time_axis_support, "time_axis_support.csv")

sparse_time_bins = time_axis_support[time_axis_support["dropped_before_analysis"]].copy()
save_table(sparse_time_bins, "sparse_time_bins_dropped.csv")

valid_bins = set(time_axis_support.loc[time_axis_support["meets_min_reviews"], "time_bin"])
subset_df = subset_df[subset_df["time_bin"].isin(valid_bins)].copy()

time_order = natural_sort_time_bins(subset_df["time_bin"].unique())
if len(time_order) < 2:
    raise ValueError(
        "Fewer than two supported time bins remain after filtering. Lower MIN_REVIEWS_PER_TIME_BIN, "
        "increase MAX_REVIEWS_TOTAL, choose a broader category, use quarter/year TIME_GRANULARITY, "
        "or inspect sparse_time_bins_dropped.csv and time_axis_support.csv."
    )

save_table(subset_df, "temporal_review_subset.csv")

print("Final temporal subset:", subset_df.shape)
print("Full time axis bins:", len(complete_time_order), complete_time_order[:3], "...", complete_time_order[-3:])
print("Supported time bins:", len(time_order), time_order[:3], "...", time_order[-3:])
print("Sparse or missing bins shown as gaps:", int(time_axis_support["dropped_before_analysis"].sum()))
display(subset_df.head())


## 4. Temporal EDA

Before measuring temporal “change,” inspect review volume and rating coverage. Review-volume changes can reflect actual demand, Yelp adoption, sampling, restaurant openings/closures, or data collection artifacts.

In [ ]:
def _time_axis_status_series(labels):
    if "time_axis_support" not in globals() or time_axis_support.empty:
        return pd.Series(["supported"] * len(labels), index=labels)
    status = time_axis_support.set_index("time_bin")["time_axis_status"]
    return pd.Series(labels, index=labels).map(status).fillna("supported")


def shade_unsupported_bins(ax, labels):
    statuses = _time_axis_status_series(labels)
    for idx, status in enumerate(statuses.tolist()):
        if status == "sparse_below_threshold":
            ax.axvspan(idx - 0.5, idx + 0.5, color="#f2c94c", alpha=0.18, linewidth=0)
        elif status == "missing_from_raw_subset":
            ax.axvspan(idx - 0.5, idx + 0.5, color="#d9d9d9", alpha=0.35, linewidth=0)


def set_time_axis_ticks(ax, labels, max_ticks=14):
    labels = list(labels)
    if not labels:
        return
    step = max(1, int(np.ceil(len(labels) / max_ticks)))
    ticks = list(range(0, len(labels), step))
    if ticks[-1] != len(labels) - 1:
        ticks.append(len(labels) - 1)
    ax.set_xticks(ticks)
    ax.set_xticklabels([labels[i] for i in ticks], rotation=45, ha="right")




### Reading temporal visualizations: shading conventions

The helper functions above define a consistent visual language used in every plot in this notebook:

- **Yellow shading** (`#f2c94c`, alpha 0.18) — sparse bin: reviews exist but below `MIN_REVIEWS_PER_TIME_BIN`. The plotted value is based on fewer observations than usual; treat it as approximate.
- **Gray shading** (`#d9d9d9`, alpha 0.35) — missing bin: no reviews for this category/time combination. No value is plotted; the x-axis tick appears as a gap.
- **The red dashed line** on the volume chart — the `MIN_REVIEWS_PER_TIME_BIN` threshold. Bars below this line are the sparse bins shown in yellow on other charts.

> **Student insight:** your report should never describe a trend that "crosses" a large gray gap without explicitly noting the gap. A rising trend that disappears for three years and then resumes is not the same as a steady rise — it may reflect platform adoption, data coverage, or a genuine gap in business activity.

Now build the temporal summary table and the first two diagnostic plots.

In [ ]:
# TODO (Category E — Guided): Build temporal review summary and rating-over-time table.
# Estimated time: ~0.5 h
#
# Expected inputs:
#   subset_df         : DataFrame with columns review_id, time_bin, stars, business_id
#   time_axis_support : DataFrame [time_bin, supported (bool), review_count, ...]
#   observed_time_order : sorted list of time bin labels
#
# Steps:
# 1. temporal_supported = subset_df merged with time_axis_support; filter supported==True.
# 2. For each time_bin: count reviews (review_count), count unique businesses,
#    compute avg_stars. Save as review_counts_and_ratings_by_time.csv.
# 3. Build temporal_summary: per time_bin — review_count, unique_businesses, avg_stars,
#    meets_min_reviews (bool: review_count >= MIN_REVIEWS_PER_BIN).

# TODO: build temporal_supported
plot_labels = complete_time_order if complete_time_order else observed_time_order
time_axis_support["supported"] = time_axis_support["meets_min_reviews"]
temporal_supported = subset_df.merge(
    time_axis_support[["time_bin", "supported"]],
    on="time_bin",
    how="left",
)
temporal_supported = temporal_supported[temporal_supported["supported"].fillna(False)].copy()

# TODO: aggregate review_counts_and_ratings_by_time
review_counts_and_ratings = (
    temporal_supported.groupby("time_bin")
    .agg(
        review_count=("review_id", "nunique"),
        unique_businesses=("business_id", "nunique"),
        avg_stars=("stars", "mean"),
    )
    .reset_index()
)

# TODO: build temporal_summary
temporal_summary = time_axis_support[[
    "time_bin", "review_count", "meets_min_reviews", "time_axis_status"
]].merge(
    review_counts_and_ratings[["time_bin", "unique_businesses", "avg_stars"]],
    on="time_bin",
    how="left",
)
temporal_summary["unique_businesses"] = temporal_summary["unique_businesses"].fillna(0).astype(int)
temporal_summary["review_count"] = temporal_summary["review_count"].fillna(0).astype(int)
review_counts_and_ratings = temporal_summary[[
    "time_bin", "review_count", "unique_businesses", "avg_stars",
    "meets_min_reviews", "time_axis_status",
]].copy()
save_table(review_counts_and_ratings, "review_counts_and_ratings_by_time.csv")
print(f"✓ temporal_summary: {len(temporal_summary)} time bins, {temporal_summary['meets_min_reviews'].sum()} supported")
display(review_counts_and_ratings.head())

# --- Validation ---
assert temporal_summary is not None and "time_bin" in temporal_summary.columns
save_table(review_counts_and_ratings, "review_counts_and_ratings_by_time.csv")
print(f"\u2713 temporal_summary: {len(temporal_summary)} time bins, {temporal_summary['meets_min_reviews'].sum()} supported")


### Review volume: your first and most important temporal check

Before interpreting any trend, look at this bar chart. Ask yourself:

1. **Is volume roughly stable across supported bins?** If review counts grow substantially over time (as they do on Yelp — platform growth drove a ~10× increase from 2008 to 2018), any raw count of dish mentions will appear to "rise" even if the *rate* is flat or falling. Section 6 normalizes dish mentions by review volume specifically to correct for this.

2. **Where are the gaps and sparse bins?** A sparse bin early in the time axis (e.g., 2008Q1 with only 3 reviews) is expected and harmless. A sparse bin in the middle of an otherwise-dense period warrants investigation.

3. **Is this category well-represented throughout?** Some cuisine categories are recent arrivals on Yelp. Chinese food has been on the platform since 2005; some specialty categories only accumulate enough reviews after 2015. If more than 30% of your bins are sparse or missing, consider widening the category gate or switching to `year` granularity.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
avg_for_plot = temporal_summary["avg_stars"].where(temporal_summary["meets_min_reviews"])
rolling = avg_for_plot.rolling(ROLLING_WINDOW, min_periods=1).mean().where(temporal_summary["meets_min_reviews"])
shade_unsupported_bins(ax, plot_labels)
ax.plot(np.arange(len(plot_labels)), avg_for_plot, marker="o", label="Average stars")
if len(temporal_summary) >= ROLLING_WINDOW:
    ax.plot(np.arange(len(plot_labels)), rolling, linestyle="--", label=f"{ROLLING_WINDOW}-bin rolling mean")
ax.set_title(f"Average rating over time with sparse/missing gaps - {TARGET_CATEGORY}")
ax.set_xlabel("Time bin")
ax.set_ylabel("Average stars")
ax.set_ylim(1, 5)
set_time_axis_ticks(ax, plot_labels)
ax.legend()
save_figure("average_rating_over_time.png")
plt.show()


### Average rating over time: interpretation pitfalls

The rolling-mean line smooths quarter-to-quarter noise, but it can also hide important short-term events. Keep these in mind:

- **Rating inflation**: Yelp average ratings tend to be slightly higher in recent years across many categories. A rising average may reflect real improvement, or it may reflect platform-wide term shift in reviewer calibration. Section 9 (term shift) can help you check whether the *language* of reviews changed in parallel.
- **Volume confounding**: a quarter with three reviews at 4.5★ looks the same as a quarter with 500 reviews at 4.5★ on this chart. Always read the rating chart alongside the volume chart.
- **Gaps break rolling means**: the `ROLLING_WINDOW`-bin rolling average is only computed over supported bins. A rolling mean of 2 is conservative; increase it only if you have dense, mostly-supported coverage.

> **Student task:** identify the quarter with the lowest average rating in your dataset. Is it supported (enough reviews)? Does the text in Section 10 (notable-period examples) match what you would expect from a low-rating period?

## 5. Load or construct a dish list

Task 5 reuses the curated `final_dish_list.csv` from Task 3 and, when available, the Task 4 `dish_mention_coverage.csv` artifact. The Task 3 artifact supplies the vocabulary; the Task 4 artifact helps identify which dishes were supported enough for evidence-backed restaurant comparison. If those artifacts are unavailable, the setup smoke path uses a small fallback list.

The key rule is the same as Task 4: temporal dish trends are only meaningful for phrases that have enough evidence. Task 5 does not blindly trend every candidate; it carries forward readiness and coverage signals when they exist.

Task 5 treats the dish list as a curated handoff from Task 3. Auto-generated candidates should not be used just because they were discovered algorithmically; if an automatically discovered dish has been human-reviewed, mark it with a curated status such as `approved_after_review` or `task4_ready`.

### Why the dish list comes from Task 3

This notebook does not discover dishes from scratch — it *monitors* dishes that were already curated in Task 3. This is intentional:

- **Precision over recall**: Task 3 filtered out generic food words, fragments, and business names. Using its output here means your temporal analysis tracks real dish vocabulary, not noise.
- **Continuity across tasks**: the dish vocabulary in Task 3 → Task 4 → Task 5 forms a single analytical thread. If you used a different dish list in Task 5, you could not compare your findings to Task 4 rankings.
- **Task 4 evidence tiers inform priority**: dishes rated `strong` or `moderate` in Task 4's `dish_mention_coverage.csv` have enough review evidence to produce stable temporal trends. Dishes rated `sparse` or `not_rankable` are loaded but flagged — their temporal trends will be noisy.

> If `artifacts/task3/final_dish_list.csv` does not exist, a short hardcoded fallback list is used. This is acceptable for smoke testing but **not** for a real submission. Your report should state which source was used.

In [ ]:
TASK3_DISH_PATH = Path(os.environ.get("TASK5_DISH_LIST_PATH", "artifacts/task3/final_dish_list.csv"))
TASK4_COVERAGE_PATH = Path(os.environ.get("TASK5_TASK4_COVERAGE_PATH", "artifacts/task4/dish_mention_coverage.csv"))

def load_dish_list_for_task5():
    if TASK3_DISH_PATH.exists():
        dish_df = pd.read_csv(TASK3_DISH_PATH)
        # tolerate several possible schemas from student/instructor Task 3
        if "dish" not in dish_df.columns:
            raise ValueError(f"{TASK3_DISH_PATH} exists but has no 'dish' column.")
        if "curation_status" in dish_df.columns:
            approved = dish_df["curation_status"].astype(str).str.lower().isin(
                [
                    "approved", "approve", "yes", "task4_ready", "approved_after_review",
                    "approved_seed", "human_approved", "curated",
                ]
            )
            dish_df = dish_df[approved].copy()
        if "task4_ready" in dish_df.columns:
            ready = dish_df["task4_ready"].astype(str).str.lower().isin(["true", "1", "yes", "y"])
            if ready.any():
                dish_df = dish_df[ready].copy()
        source = "task3_artifact"
    else:
        fallback = [
            "dumplings", "noodles", "fried rice", "hot pot", "mapo tofu",
            "kung pao chicken", "spring rolls"
        ]
        dish_df = pd.DataFrame({
            "dish": fallback,
            "cuisine_or_category": TARGET_CATEGORY,
            "source_method": "smoke_or_instructor_fallback",
            "curation_status": "approved_after_review",
        })
        source = "fallback"

    dish_df["dish"] = dish_df["dish"].astype(str).str.strip().str.lower()
    dish_df = dish_df[dish_df["dish"].str.len() > 0].drop_duplicates("dish").copy()

    if TASK4_COVERAGE_PATH.exists():
        coverage = pd.read_csv(TASK4_COVERAGE_PATH)
        if "dish" in coverage.columns:
            coverage = coverage.copy()
            coverage["dish"] = coverage["dish"].astype(str).str.strip().str.lower()
            keep = [c for c in ["dish", "mention_count", "businesses_with_mentions", "evidence_tier", "main_analysis_ready"] if c in coverage.columns]
            coverage = coverage[keep].drop_duplicates("dish")
            rename_map = {c: f"task4_{c}" for c in keep if c != "dish"}
            dish_df = dish_df.merge(coverage.rename(columns=rename_map), on="dish", how="left")
            source += "+task4_coverage"
    if "task4_evidence_tier" not in dish_df.columns:
        dish_df["task4_evidence_tier"] = "not_available"
    if "task4_main_analysis_ready" not in dish_df.columns:
        dish_df["task4_main_analysis_ready"] = pd.NA

    tier_order = {"strong": 0, "moderate": 1, "sparse": 2, "not_rankable": 3, "not_available": 4}
    dish_df["_task4_tier_order"] = dish_df["task4_evidence_tier"].fillna("not_available").map(tier_order).fillna(4)
    dish_df = dish_df.sort_values(["_task4_tier_order", "dish"]).drop(columns=["_task4_tier_order"]).head(MAX_DISHES).copy()
    if dish_df.empty:
        raise ValueError(
            "No usable curated dishes loaded for Task 5 temporal dish analysis. "
            "Use Task 3 statuses such as approved, approved_after_review, task4_ready, or approved_seed; "
            "if an auto-generated dish was human-reviewed, rename its status to approved_after_review before using it here."
        )
    return dish_df, source

dish_df, dish_source = load_dish_list_for_task5()
save_table(dish_df, "task5_dish_list_used.csv")
save_table(dish_df, "task5_upstream_artifact_handoff.csv")
print("Dish source:", dish_source)
display(dish_df)

## Data Mining Concept: Temporal Pattern Discovery — Separating Real Change from Sampling Artifacts

**The temporal data mining question:** How do restaurant review patterns change over time? Are certain dishes becoming more popular? Is sentiment improving or declining? Are reviews using different vocabulary in recent years? Are these changes real, or are they artifacts of who chose to write reviews in each period?

Task 5 uses three safeguards:

1. **Normalize by review volume.** A dish mention count is not meaningful unless you know how many reviews existed in that period.
2. **Require support.** Sparse bins are shown but treated cautiously.
3. **Triangulate.** Compare volume, ratings, dish mention rates, sentiment, distinctive terms, and actual review excerpts before making a temporal claim.

This keeps the task focused on temporal evidence mining rather than forecasting or causal inference.


## 6. Temporal dish mention trends

Raw mention counts are useful, but they can be misleading when review volume changes. We therefore compute both:

- raw dish mention counts by time bin;
- normalized mention rates per 1,000 reviews.

A dish-time cell can be useful for exploration while still being too sparse for confident trend interpretation, so the notebook flags whether each dish/time bin meets a minimum mention-review threshold.

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def phrase_pattern(phrase):
    phrase_norm = re.escape(normalize_text(phrase))
    phrase_norm = phrase_norm.replace("\\ ", r"\s+")
    return re.compile(rf"(?<![a-z0-9]){phrase_norm}(?![a-z0-9])", flags=re.IGNORECASE)

subset_df["text_norm"] = subset_df["text"].map(normalize_text)
dish_patterns = {dish: phrase_pattern(dish) for dish in dish_df["dish"].tolist()}

mention_rows = []
for idx, row in subset_df.iterrows():
    text_norm = row["text_norm"]
    for dish, pat in dish_patterns.items():
        matches = list(pat.finditer(text_norm))
        if not matches:
            continue
        mention_rows.append({
            "review_id": row["review_id"],
            "business_id": row["business_id"],
            "business_name": row.get("name"),
            "date": row["date"],
            "time_bin": row["time_bin"],
            "stars": row["stars"],
            "dish": dish,
            "mention_count_in_review": len(matches),
            "text": row["text"],
        })



### From reviews to mentions: the phrase matching step

The `phrase_pattern` function builds a regex that:
- **Word-boundary anchors** (`(?<![a-z0-9])...(?![a-z0-9])`) ensure `"noodles"` does not match inside `"noodle soup"` the wrong way, and that partial matches (e.g., `"hot"` inside `"shot"`) are excluded.
- **`\s+` for spaces** allows multi-word dish names to match even when reviewers write `"hot  pot"` (double space) or include punctuation variations.
- **`normalize_text` preprocessing** lowercases and removes most punctuation before matching, so the pattern works uniformly.

> **What this means for your results:** the mention count depends heavily on the exact dish strings in your Task 3 list. A dish called `"dim sum"` will not match reviews that say `"dim-sum"` unless the normalization handles hyphens. The normalization above converts hyphens to spaces, so `"dim-sum"` → `"dim sum"` → match. But it will not match `"dimsum"` (no space). Inspect `dish_mentions_by_time_long.csv` for edge cases.

The loop below runs every review against every dish pattern. For `MAX_REVIEWS_TOTAL=10,000` and `MAX_DISHES=12` this is fast (< 5 seconds). For larger subsets, consider vectorizing with string.contains or building a combined regex.

In [ ]:
# TODO (Category E — Guided): Aggregate dish mentions by time bin.
# Estimated time: ~0.5 h
#
# Expected input: mentions_df with columns [review_id, time_bin, dish, mention_count_in_review,
#   business_id, stars, text]
#
# Step 1: Build dish_counts by groupby ["time_bin", "dish"]:
#   mention_reviews  = nunique of review_id
#   total_mentions   = sum of mention_count_in_review
#   businesses_with_mentions = nunique of business_id
#   avg_stars_for_mentions  = mean of stars
#
# Step 2: Compute review_counts_by_bin = subset_df.groupby("time_bin")["review_id"].nunique()
# Step 3: Merge dish_counts with review_counts_by_bin; compute:
#   mention_rate = mention_reviews / total_reviews_in_bin * 1000  (per-1000 reviews)
#
# Step 4: Build rate_wide (pivot): index=dish, columns=time_bin, values=mention_rate.
#   Reorder columns by observed_time_order.
# Step 5: Save dish_mentions_by_time_long.csv.
#
# Required output variables: dish_counts, rate_wide

# TODO: build dish_counts by aggregating mentions_df
mention_columns = [
    "review_id", "business_id", "business_name", "date", "time_bin",
    "stars", "dish", "mention_count_in_review", "text",
]
mentions_df = pd.DataFrame(mention_rows, columns=mention_columns)

if not mentions_df.empty:
    aggregated_dish_counts = (
        mentions_df.groupby(["time_bin", "dish"])
        .agg(
            mention_reviews=("review_id", "nunique"),
            total_mentions=("mention_count_in_review", "sum"),
            businesses_with_mentions=("business_id", "nunique"),
            avg_stars_for_mentions=("stars", "mean"),
        )
        .reset_index()
    )
else:
    aggregated_dish_counts = pd.DataFrame(columns=[
        "time_bin", "dish", "mention_reviews", "total_mentions",
        "businesses_with_mentions", "avg_stars_for_mentions",
    ])

dish_time_grid = pd.MultiIndex.from_product(
    [time_order, dish_df["dish"].tolist()],
    names=["time_bin", "dish"],
).to_frame(index=False)
dish_counts = dish_time_grid.merge(
    aggregated_dish_counts,
    on=["time_bin", "dish"],
    how="left",
)
for count_column in ["mention_reviews", "total_mentions", "businesses_with_mentions"]:
    dish_counts[count_column] = dish_counts[count_column].fillna(0).astype(int)

# TODO: compute mention_rate and pivot to rate_wide
review_counts_by_bin = (
    subset_df.groupby("time_bin")["review_id"]
    .nunique()
    .rename("total_reviews_in_bin")
    .reset_index()
)
dish_counts = dish_counts.merge(review_counts_by_bin, on="time_bin", how="left")
dish_counts["mention_rate"] = np.where(
    dish_counts["total_reviews_in_bin"].fillna(0) > 0,
    dish_counts["mention_reviews"] / dish_counts["total_reviews_in_bin"] * 1000,
    np.nan,
)
rate_wide = dish_counts.pivot(
    index="dish", columns="time_bin", values="mention_rate"
).reindex(
    index=dish_df["dish"].tolist(),
    columns=observed_time_order,
)
# --- Validation ---
assert isinstance(dish_counts, pd.DataFrame) and "mention_reviews" in dish_counts.columns
assert isinstance(rate_wide, pd.DataFrame)
save_table(
    dish_counts.rename(
        columns={"mention_rate": "mention_reviews_per_1000_reviews"}
    ),
    "dish_mentions_by_time_long.csv",
)
print(f"\u2713 rate_wide: {rate_wide.shape} ({rate_wide.shape[0]} dishes × {rate_wide.shape[1]} time bins)")


### Normalizing mention counts: why raw counts mislead

`dish_counts["mention_reviews_per_1000_reviews"]` divides mention reviews by the total reviews in that quarter and multiplies by 1,000. This is *not* optional.

**Without normalization**, a dish with 10 mentions in a quarter with 50 reviews (200 per 1,000) looks weaker than a dish with 12 mentions in a quarter with 1,000 reviews (12 per 1,000) — even though the first dish is far more prominent. This matters because Yelp review volume grew substantially over time; any raw trend from 2010 to 2020 will appear to "rise" on a count basis.

The `sufficient_dish_bin_evidence` flag marks cells where `mention_reviews >= MIN_DISH_MENTION_REVIEWS_PER_BIN`. Cells below this threshold are included in wide-form tables but should not be the basis for claims in your report. The heatmap in Section 6 uses `NaN` for unsupported bins so they appear as gray cells — a visual reminder to check evidence support before interpreting a pattern.

In [ ]:
fig, ax = plt.subplots(figsize=(max(10, len(rate_wide.columns) * 0.45), max(4, len(rate_wide) * 0.45)))
masked_values = np.ma.masked_invalid(rate_wide.values.astype(float))
cmap = plt.cm.viridis.copy()
cmap.set_bad("#eeeeee")
im = ax.imshow(masked_values, aspect="auto", cmap=cmap)
fig.colorbar(im, ax=ax, label="Mention reviews per 1,000 reviews")
ax.set_yticks(np.arange(len(rate_wide.index)))
ax.set_yticklabels(rate_wide.index)
set_time_axis_ticks(ax, rate_wide.columns.tolist(), max_ticks=16)
ax.set_title(f"Normalized dish mention rates over complete time axis - {TARGET_CATEGORY}")
ax.set_xlabel("Time bin")
ax.set_ylabel("Dish")
save_figure("dish_mentions_heatmap.png")
plt.show()

# Simple rising/falling calculation: compare first half vs second half over the full axis, ignoring gap NaNs.
axis_columns = list(rate_wide.columns)
mid = max(1, len(axis_columns) // 2)
early_bins = axis_columns[:mid]
late_bins = axis_columns[mid:]
trend_rows = []
for dish in rate_wide.index:
    early = rate_wide.loc[dish, early_bins].mean(skipna=True) if early_bins else np.nan
    late = rate_wide.loc[dish, late_bins].mean(skipna=True) if late_bins else np.nan
    trend_rows.append({
        "dish": dish,
        "early_avg_rate": early,
        "late_avg_rate": late,
        "late_minus_early": late - early,
    })
dish_trends = pd.DataFrame(trend_rows).sort_values("late_minus_early", ascending=False)
save_table(dish_trends, "dish_rising_falling_summary.csv")
display(dish_trends)


## 7. Temporal rating and lightweight sentiment trends

Task 4 used dish-local sentiment for recommendation. Task 5 uses a lighter temporal signal because the focus is not sentiment modeling itself; the focus is **change over time**.

Task 5 sentiment is intentionally lightweight and review-level. For precise dish-local evidence-backed ranking, see Task 4; here the goal is aggregate temporal pattern discovery, not individual dish recommendation.

We compute:

- average stars by time bin;
- a simple lexicon sentiment score by time bin;
- optional dish-mention sentiment trends.

In [ ]:
# TODO (Category B — Classic data mining): Implement lexicon-based temporal sentiment.
# Estimated time: ~1.5 h
#
# Step 1 — Define POSITIVE_WORDS and NEGATIVE_WORDS sets (dish/food domain).
#   Use 12–20 tokens per set. Think about what "positive" and "negative" mean
#   specifically in restaurant reviews about dishes.
#
# Step 2 — Implement simple_lexicon_sentiment(text) → float in [-1, 1].
#   Tokenize with re.findall(r"[a-z']+", text.lower()).
#   score = (pos_count - neg_count) / max(1, len(tokens))
#   Clamp to [-1, 1].
#
# Step 3 — Implement window_lexicon_sentiment(text) → float.
#   Applied to a dish-local context window (a sentence or short paragraph).
#   Can delegate to simple_lexicon_sentiment or add window-specific logic.
#
# Note: this reuses the same concept from Task 4 but in a temporal context.
# The goal is to see whether dish sentiment changes over time.

POSITIVE_WORDS = {
    "amazing", "awesome", "balanced", "best", "crisp", "crispy",
    "delicious", "excellent", "favorite", "flavorful", "fresh",
    "friendly", "great", "juicy", "love", "perfect", "rich",
    "tasty", "tender", "worth",
}
NEGATIVE_WORDS = {
    "awful", "bad", "bland", "cold", "disappointing", "dry",
    "expensive", "greasy", "mediocre", "overcooked", "overpriced",
    "salty", "slow", "soggy", "stale", "terrible", "undercooked",
    "underwhelming", "worst", "wrong",
}

def simple_lexicon_sentiment(text: str) -> float:
    tokens = re.findall(r"[a-z']+", str(text).lower())
    pos_count = sum(token in POSITIVE_WORDS for token in tokens)
    neg_count = sum(token in NEGATIVE_WORDS for token in tokens)
    score = (pos_count - neg_count) / max(1, len(tokens))
    return float(np.clip(score, -1.0, 1.0))

def window_lexicon_sentiment(text: str) -> float:
    # TODO: can delegate to simple_lexicon_sentiment; add window-specific adjustments if desired
    return simple_lexicon_sentiment(text)

# --- Validation ---
assert simple_lexicon_sentiment("excellent crispy delicious") > 0
assert simple_lexicon_sentiment("bland cold disappointing") < 0
print("\u2713 Lexicon sentiment helpers ready")




### Lexicon sentiment: what it captures and what it misses

The lexicon above uses 16 positive and 14 negative anchor words. This is a very coarse classifier — it treats every occurrence of "great" the same regardless of context ("great dim sum" vs "not so great anymore"). Keep this in mind when reading the results:

- **Negation blindness**: `"not great"` scores positively because only `"great"` is counted. VADER (used in Task 4) handles negation; this lexicon does not.
- **Length bias**: a 300-word review with two positive and two negative words scores 0.0. A 20-word review with one positive and zero negative words scores 1.0. Long reviews from experienced reviewers often score closer to 0.
- **Temporal stability**: if reviewers in 2010 wrote shorter, simpler reviews ("great food, great service"), the average sentiment will be systematically higher in early periods even if quality was the same. Compare with the star-rating trend to check for this.

> **Why use this at all?** It is transparent, reproducible, and computable without any model. It gives you a temporal signal that you can explain fully in a report. Task 4 showed the transformer comparison; this task shows whether the *temporal pattern* is robust enough to survive even a naive classifier.

The dish-level sentiment computation below is the more analytically useful version: it computes sentiment *only for reviews that mention each dish*, so you can track whether the mood around a specific dish is improving or declining over time.

In [ ]:
# TODO (Category E — Guided): Compute lexicon sentiment per mention and aggregate over time.
# Estimated time: ~0.5 h
#
# Step 1: Add a lexicon_sentiment column to mentions_df:
#   mentions_df["lexicon_sentiment"] = mentions_df["text"].map(simple_lexicon_sentiment)
#
# Step 2: Build dish_sentiment by groupby ["time_bin", "dish"]:
#   mention_reviews     = nunique of review_id
#   avg_stars           = mean of stars
#   avg_lexicon_sentiment = mean of lexicon_sentiment
#
# Step 3: Build sentiment_trends: groupby "time_bin" across all dishes/reviews:
#   avg_lexicon_sentiment = mean
#   avg_stars = mean
#
# Step 4: Save dish_sentiment_by_time.csv for dish-level sentiment and
#   temporal_sentiment_trends.csv for overall time-bin sentiment.

if not mentions_df.empty:
    # TODO: add lexicon_sentiment column
    # TODO: build dish_sentiment
    mentions_df = mentions_df.copy()
    mentions_df["lexicon_sentiment"] = mentions_df["text"].map(simple_lexicon_sentiment)
    dish_sentiment = (
        mentions_df.groupby(["time_bin", "dish"])
        .agg(
            mention_reviews=("review_id", "nunique"),
            avg_stars=("stars", "mean"),
            avg_lexicon_sentiment=("lexicon_sentiment", "mean"),
        )
        .reset_index()
    )
    dish_sentiment["meets_min_mention_reviews"] = dish_sentiment["mention_reviews"] >= MIN_DISH_MENTION_REVIEWS_PER_BIN
    dish_sentiment = dish_sentiment[dish_sentiment["meets_min_mention_reviews"]].copy()
    # TODO: build sentiment_trends
    sentiment_trends = (
        mentions_df.groupby("time_bin")
        .agg(
            avg_lexicon_sentiment=("lexicon_sentiment", "mean"),
            avg_stars=("stars", "mean"),
        )
        .reset_index()
    )
    # save_table(dish_sentiment, "dish_sentiment_by_time.csv")
    save_table(dish_sentiment, "dish_sentiment_by_time.csv")
    # save_table(sentiment_trends, "temporal_sentiment_trends.csv")
    save_table(sentiment_trends, "temporal_sentiment_trends.csv")
    # print(f"\u2713 dish_sentiment: {len(dish_sentiment)} rows")
    print(f"✓ dish_sentiment: {len(dish_sentiment)} rows")
else:
    dish_sentiment = pd.DataFrame(columns=["time_bin", "dish", "mention_reviews", "avg_stars", "avg_lexicon_sentiment", "meets_min_mention_reviews"])
    sentiment_trends = pd.DataFrame(columns=["time_bin", "avg_lexicon_sentiment", "avg_stars"])
    print("Warning: mentions_df is empty; dish sentiment will be empty.")


### Reading `dish_sentiment_by_time.csv`

This table has one row per (dish, time_bin) pair with enough mention reviews. Compare `avg_lexicon_sentiment` against `avg_stars_for_mentions` from `dish_mentions_by_time_long.csv`. If sentiment and stars move in opposite directions for a dish, that is a meaningful discrepancy worth investigating with Section 10 (notable-period review examples).

> **Student task:** pick one dish from your list and describe its sentiment trajectory across time. Is it consistent with the normalized mention rate trend from Section 6? If sentiment falls while mention rate rises, what might explain that pattern?

## 8. Temporal term trends

This section asks: which terms are most distinctive in each time bin?

This is a lightweight alternative to fitting a separate topic model for every time window. Separate per-window topic models can be unstable when time bins have few reviews.

The custom stopword list is for topical term trends, not sentiment analysis. It intentionally removes generic restaurant-review language and the selected target-category label so the heatmap is more likely to surface cuisine, dish, service, or period-specific language.

## About Temporal TF-IDF Shift Analysis

Temporal TF-IDF shift analysis compares which terms are distinctive in different time bins. In Task 5, students aggregate review text by period, vectorize each period, and inspect changing top terms or heatmaps to understand how review language evolves. This is a classical text-mining approach to temporal term shift because it keeps evidence tied to visible words and phrases.

The method is interpretable but sensitive to sparse bins, generic restaurant vocabulary, and vocabulary changes caused by sampling rather than real-world behavior. You should compare term shifts with review volume, dish mentions, sentiment, and example reviews before making claims.

**Key Resources:**
- [scikit-learn TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) - TF-IDF implementation reference.
- [scikit-learn Feature Extraction](https://scikit-learn.org/stable/modules/feature_extraction.html) - Text feature extraction guide.
- [Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/) - Classical term weighting reference.

In [ ]:
RESTAURANT_REVIEW_STOP_WORDS = {
    "food", "place", "restaurant", "restaurants", "order", "ordered",
    "just", "really", "like", "got", "get", "go", "went", "come", "came",
    "time", "try", "tried", "eat", "eating", "menu", "table", "people",
    "service", "staff", "location", "experience", "thing", "things", "way",
    "little", "lot", "bit", "nice", "good", "great", "best", "ve", "don",
}
TARGET_CATEGORY_STOP_WORDS = set(re.findall(r"[a-z0-9]+", normalize_text(TARGET_CATEGORY)))
TFIDF_STOP_WORDS = sorted(set(ENGLISH_STOP_WORDS) | RESTAURANT_REVIEW_STOP_WORDS | TARGET_CATEGORY_STOP_WORDS)
time_text = (
    subset_df.groupby("time_bin")["text"]
    .apply(lambda texts: " ".join(map(str, texts)))
    .reindex(time_order)
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=TFIDF_STOP_WORDS,
    ngram_range=(1, 2),
    min_df=1,
    max_features=400,
)
X_tfidf = vectorizer.fit_transform(time_text.values)
terms = np.array(vectorizer.get_feature_names_out())



### What TF-IDF "distinctiveness" means in a temporal context

TF-IDF weight for a term in a time bin = `(term frequency in this bin) × log(total bins / bins where term appears)`.

A term scores high in a time bin if it appears frequently *in that bin* but rarely *across all bins combined*. This means:

- **Generic terms that appear in every period** (e.g., "service", "good") get near-zero IDF weight and are effectively filtered out, even without appearing in the stop-word list.
- **A term that spikes in one or two periods** gets high TF-IDF in those periods, making it "distinctive" to that time window.
- **Stop words** remove the most obvious generic restaurant vocabulary. The custom `RESTAURANT_REVIEW_STOP_WORDS` set removes terms that are highly frequent across all Yelp restaurant reviews but not specific to any cuisine or time period.

> **Pay attention:** a term appearing in the heatmap's top row is not necessarily "important" in any absolute sense — it is merely the least common globally and most concentrated locally. A distinctive term for 2020Q2 might be "delivery" or "takeout" — a genuine signal of COVID-19 platform behavior, not a change in the cuisine itself. Before claiming a term trend reflects a food-quality change, consider whether it might reflect platform, review, or sampling behavior instead.

In [ ]:
# TODO (Category B — Classic data mining): Build top-terms-by-time using TF-IDF.
# Estimated time: ~2 h
#
# Background: RESTAURANT_REVIEW_STOP_WORDS and time_text (Series: time_bin → aggregated text)
# are already defined above. TF-IDF is fitted on time_text using TfidfVectorizer.
#
# Step 1: Fit TfidfVectorizer on time_text values:
#   - Use RESTAURANT_REVIEW_STOP_WORDS, MAX_FEATURES=200, min_df=1, sublinear_tf=True.
#   - Fit on [time_text.get(tb, "") for tb in observed_time_order].
#   - X_tfidf = transformed matrix, shape (n_time_bins, vocab_size).
#   - terms = vectorizer.get_feature_names_out()
#
# Step 2: For each time bin (row), find the top 12 terms by TF-IDF score.
#   Build top_rows: [{"time_bin": ..., "rank": 1..12, "term": ..., "tfidf": ...}].
#   Save to top_terms_by_time.csv.
#
# Step 3: Plot a temporal term heatmap (top 20 terms by total TF-IDF across all bins).
#   Rows = terms, columns = time bins, cell color = TF-IDF score.
#   Save to ARTIFACT_DIR / "top_terms_by_time_heatmap.png".
#   Hint: use rate_wide.T to get the right orientation.
#
# Step 4: Display top_terms_df (head 40) and the heatmap.

# TODO: fit TfidfVectorizer on time_text
time_bins_for_tfidf = observed_time_order
tfidf_documents = [
    "" if pd.isna(time_text.get(tb, "")) else str(time_text.get(tb, ""))
    for tb in time_bins_for_tfidf
]
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=sorted(set(ENGLISH_STOP_WORDS) | RESTAURANT_REVIEW_STOP_WORDS | TARGET_CATEGORY_STOP_WORDS),
    ngram_range=(1, 2),
    max_features=200,
    min_df=1,
    sublinear_tf=True,
)
X_tfidf = vectorizer.fit_transform(tfidf_documents)
terms = np.array(vectorizer.get_feature_names_out())
tfidf_array = X_tfidf.toarray()

# TODO: extract top 12 terms per time bin → top_rows list → top_terms_df
top_rows = []
for row_idx, time_bin in enumerate(time_bins_for_tfidf):
    scores = tfidf_array[row_idx]
    top_idx = np.argsort(scores)[::-1][:12]
    for rank, term_idx in enumerate(top_idx, start=1):
        if scores[term_idx] <= 0:
            continue
        top_rows.append({
            "time_bin": time_bin,
            "rank": rank,
            "term": terms[term_idx],
            "tfidf": float(scores[term_idx]),
        })
top_terms_df = pd.DataFrame(top_rows)
save_table(top_terms_df, "top_terms_by_time.csv")

# TODO: plot temporal term heatmap and save
term_matrix = pd.DataFrame(tfidf_array.T, index=terms, columns=time_bins_for_tfidf)
top_heatmap_terms = term_matrix.sum(axis=1).sort_values(ascending=False).head(20).index
term_heatmap = term_matrix.loc[top_heatmap_terms]
fig, ax = plt.subplots(
    figsize=(max(10, len(term_heatmap.columns) * 0.45), max(4, len(term_heatmap.index) * 0.35))
)
im = ax.imshow(term_heatmap.values, aspect="auto", cmap="magma")
ax.set_yticks(np.arange(len(term_heatmap.index)))
ax.set_yticklabels(term_heatmap.index)
set_time_axis_ticks(ax, term_heatmap.columns.tolist(), max_ticks=16)
ax.set_title(f"Distinctive review terms by time bin - {TARGET_CATEGORY}")
ax.set_xlabel("Time bin")
ax.set_ylabel("Term")
fig.colorbar(im, ax=ax, label="TF-IDF weight")
save_figure("top_terms_by_time_heatmap.png")
plt.show()
display(top_terms_df.head(40))
print(f"✓ top_terms_df: {len(top_terms_df)} rows, {top_terms_df['time_bin'].nunique()} time bins")

# --- Validation ---
assert isinstance(top_terms_df, pd.DataFrame) and "term" in top_terms_df.columns
assert (ARTIFACT_DIR / "top_terms_by_time.csv").exists()
assert (ARTIFACT_DIR / "top_terms_by_time_heatmap.png").exists()
print(f"\u2713 top_terms_df: {len(top_terms_df)} rows, {top_terms_df['time_bin'].nunique()} time bins")




### Reading the term trends heatmap

Each row is a term; each column is a time bin. Darker cells = higher TF-IDF weight in that period.

Things to look for:
1. **Column-wide brightness bursts** — a term that lights up brightly in a single column suggests a period-specific event or a coverage anomaly (if the bin has very few reviews, a few repeated terms can dominate).
2. **Gradual horizontal gradients** — a term that brightens left-to-right reflects a genuine shift in vocabulary over time. This is the most interpretable pattern.
3. **Gray columns** — these are unsupported bins (no data). The heatmap makes them visible so you know not to read into the surrounding patterns.

Cross-check your top terms against the notable-period review examples in Section 10. If a term dominates 2015Q3 in the heatmap, find an example review from that quarter and check whether the term appears there in a meaningful context.

## 10. Notable-period review inspection

Temporal plots can suggest patterns, but interpretation should return to actual reviews. This section extracts example reviews from notable periods.

### Why example reviews matter more than you might think

Every quantitative pattern in Sections 4–8 is computed from aggregate statistics. But Yelp reviews are written by individual people about specific experiences. Quantitative methods can tell you *that* something changed; example reviews can tell you *what* changed and *why*.

The `notable_bins` set below selects periods that stand out on three dimensions — review volume and sentiment. Inspecting actual reviews from these periods serves three purposes:

1. **Triangulation** — do the reviews confirm the quantitative signal? A surprising quarter that contains reviews about a new dish, a restaurant closing, or a menu change is interpretable. A surprising quarter with reviews that look completely normal suggests a statistical artifact.
2. **Counter-evidence** — sometimes a single outlier review (very long, very short, copy-paste) drives a whole quarter's aggregate signal. Example reviews surface these.
3. **Human anchoring** — your report should include at least one concrete review excerpt. Quantitative patterns without any anchoring text are harder to evaluate.

> **Student task:** for each notable period in `task5_notable_period_annotations.csv`, find the review that you think best *explains* the quantitative signal for that period. Include it in your report with a one-sentence interpretation.

In [ ]:
# TODO (Category D — Medium pipeline): Identify notable time bins and sample example reviews.
# Estimated time: ~1 h
#
# A "notable bin" is one that stands out by review volume or sentiment.
#
# Step 1 — Identify notable bins (up to 3):
#   - Bin with highest review_count (from temporal_summary).
#   - Bin with highest avg_lexicon_sentiment (from sentiment_trends, if available).
#   - Bin with lowest avg_lexicon_sentiment.
#   Deduplicate; only include supported bins (meets_min_reviews == True).
#
# Step 2 — For each notable bin, sample up to CONTEXT_SNIPPETS reviews:
#   Filter subset_df to that time_bin; sample min(CONTEXT_SNIPPETS, len) rows.
#   Collect: time_bin, review_id, business_id, stars, text[:300].
#   Store in example_rows list → example_reviews_df DataFrame.
#   Save to ARTIFACT_DIR / "notable_period_example_reviews.csv".
#
# Step 3 — Display 1–2 example reviews per notable bin.

notable_bins = set()
# TODO: add bins with highest/lowest review count and sentiment from temporal_summary / sentiment_trends
# Only include bins where meets_min_reviews is True.
CONTEXT_SNIPPETS = max(1, min(3, MAX_EXAMPLE_REVIEWS))
supported_summary = temporal_summary[temporal_summary["meets_min_reviews"]].copy()
if not supported_summary.empty:
    notable_bins.add(str(supported_summary.loc[supported_summary["review_count"].idxmax(), "time_bin"]))
if isinstance(sentiment_trends, pd.DataFrame) and not sentiment_trends.empty:
    supported_bins = set(supported_summary["time_bin"].astype(str))
    supported_sentiment = sentiment_trends[
        sentiment_trends["time_bin"].astype(str).isin(supported_bins)
    ].dropna(subset=["avg_lexicon_sentiment"])
    if not supported_sentiment.empty:
        notable_bins.add(str(supported_sentiment.loc[
            supported_sentiment["avg_lexicon_sentiment"].idxmax(), "time_bin"
        ]))
        notable_bins.add(str(supported_sentiment.loc[
            supported_sentiment["avg_lexicon_sentiment"].idxmin(), "time_bin"
        ]))

example_rows = []
for time_bin in notable_bins:
    # TODO: sample rows from subset_df for this time_bin; append to example_rows
    bin_reviews = subset_df[subset_df["time_bin"].astype(str) == str(time_bin)]
    sampled_reviews = bin_reviews.sample(
        n=min(CONTEXT_SNIPPETS, len(bin_reviews)),
        random_state=RANDOM_SEED,
    )
    for _, row in sampled_reviews.iterrows():
        example_rows.append({
            "time_bin": str(time_bin),
            "review_id": row["review_id"],
            "business_id": row["business_id"],
            "stars": row["stars"],
            "text_snippet": str(row["text"])[:300],
        })

example_reviews_df = pd.DataFrame(example_rows)
# save_table(example_reviews_df, "notable_period_example_reviews.csv")
save_table(example_reviews_df, "notable_period_example_reviews.csv")
# display(example_reviews_df.head(10))
display(example_reviews_df.groupby("time_bin").head(2))

# --- Validation ---
print(f"\u2713 Notable bins: {sorted(notable_bins)}, {len(example_reviews_df)} example reviews sampled")


## 10b. Temporal Embedding Drift Analysis

How does the *semantic content* of reviews shift over time? Whereas TF-IDF measures
changes in individual term frequencies, embedding-centroid drift measures shifts in
**overall meaning**: a large centroid distance between two bins suggests the topics,
tone, and vocabulary changed substantially.

This section encodes each time-bin's reviews into a centroid embedding (using
`sentence-transformers` or a TF-IDF + SVD fallback), then measures cosine distance
between consecutive bin centroids.

In [ ]:
# TODO (Category A — Modern library): Compute temporal embedding-centroid drift.
# Estimated time: ~2.5 h
#
# Expected inputs:
#   temporal_supported    : DataFrame with columns [time_bin, text, ...]
#   observed_time_order   : sorted list of time bin labels (strings)
#   RANDOM_SEED           : integer seed for reproducibility
#
# ── A) Encode per-bin text into centroid embeddings ─────────────────────────
#
#   Try:  from sentence_transformers import SentenceTransformer
#         model = SentenceTransformer("all-MiniLM-L6-v2")
#         For each bin: sample up to 200 sentences; encode; take mean → centroid.
#
#   Fallback (if sentence_transformers unavailable):
#         from sklearn.feature_extraction.text import TfidfVectorizer
#         from sklearn.decomposition import TruncatedSVD
#         Fit TfidfVectorizer on all texts; TruncatedSVD(n_components=64).
#         For each bin: transform sample; take mean → centroid.
#         Label representation = "tfidf_svd_fallback".
#
# ── B) Compute cosine distance between consecutive bin centroids ─────────────
#
#   from sklearn.metrics.pairwise import cosine_similarity
#   For adjacent bins (t_i, t_{i+1}):
#     drift = 1 - cosine_similarity([centroid_t_i], [centroid_t_{i+1}])[0, 0]
#   Build drift_df: columns [time_bin_from, time_bin_to, cosine_drift, representation].
#
# ── C) Visualize ────────────────────────────────────────────────────────────
#
#   Line chart: x = bin-transition label (e.g., "2018-Q4 → 2019-Q1"),
#               y = cosine_drift.
#   Annotate the two largest drift spikes with their bin-transition labels.
#   Save to ARTIFACT_DIR / "embedding_drift_over_time.png".
#
# ── D) Save and interpret ────────────────────────────────────────────────────
#
#   Save drift_df to ARTIFACT_DIR / "embedding_centroid_drift.csv".
#   In 2–4 comments, write a brief interpretation:
#     Which periods had the largest drift?
#     How does this compare to TF-IDF term-change signal from the previous section?
#     Could the drift reflect a real trend, seasonal effect, or sampling artifact?
#
# Hint: Wrap sentence-transformers import in try/except to degrade gracefully.
#       In smoke-test mode, use the fallback regardless of availability.

try:
    from sentence_transformers import SentenceTransformer as _SentenceTransformer
    _ST_AVAILABLE = True
except ImportError:
    _ST_AVAILABLE = False
    print("sentence-transformers not available; will use TF-IDF + SVD fallback.")

# TODO: encode per-bin text into centroid embeddings (ST or TF-IDF+SVD fallback)
from sklearn.decomposition import TruncatedSVD

MAX_TEXTS_PER_EMBEDDING_BIN = 200
embedding_source = temporal_supported
embedding_bins = [
    tb for tb in observed_time_order
    if tb in set(embedding_source["time_bin"].astype(str))
]
bin_text_samples = {}
for time_bin in embedding_bins:
    texts = embedding_source.loc[
        embedding_source["time_bin"].astype(str) == str(time_bin), "text"
    ].dropna().astype(str)
    if len(texts) > MAX_TEXTS_PER_EMBEDDING_BIN:
        texts = texts.sample(n=MAX_TEXTS_PER_EMBEDDING_BIN, random_state=RANDOM_SEED)
    bin_text_samples[str(time_bin)] = texts.tolist()

bin_centroids = {}
representation = "tfidf_svd_fallback"
embedding_model_path = os.environ.get("TASK5_EMBEDDING_MODEL_PATH", "").strip()
allow_embedding_download = os.environ.get("TASK5_ALLOW_EMBEDDING_DOWNLOAD", "0") == "1"
model_source = None
if embedding_model_path and Path(embedding_model_path).exists():
    model_source = embedding_model_path
elif allow_embedding_download:
    model_source = "all-MiniLM-L6-v2"

if _ST_AVAILABLE and DATASET_MODE != "smoke_test" and model_source is not None:
    try:
        embedding_model = _SentenceTransformer(model_source)
        representation = f"sentence_transformer:{model_source}"
        for time_bin, texts in bin_text_samples.items():
            if texts:
                embeddings = embedding_model.encode(
                    texts, batch_size=32, show_progress_bar=False
                )
                bin_centroids[time_bin] = np.asarray(embeddings).mean(axis=0)
    except Exception as exc:
        print(f"Sentence-transformer embedding failed; using TF-IDF + SVD fallback: {exc}")
        bin_centroids = {}
        representation = "tfidf_svd_fallback"

if not bin_centroids:
    all_embedding_texts = [text for texts in bin_text_samples.values() for text in texts]
    if all_embedding_texts:
        drift_vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words=TFIDF_STOP_WORDS,
            ngram_range=(1, 2),
            min_df=1,
            max_features=500,
            sublinear_tf=True,
        )
        drift_tfidf = drift_vectorizer.fit_transform(all_embedding_texts)
        svd_components = max(1, min(64, drift_tfidf.shape[0] - 1, drift_tfidf.shape[1] - 1))
        drift_svd = TruncatedSVD(n_components=svd_components, random_state=RANDOM_SEED)
        drift_svd.fit(drift_tfidf)
        for time_bin, texts in bin_text_samples.items():
            if texts:
                bin_tfidf = drift_vectorizer.transform(texts)
                bin_embeddings = drift_svd.transform(bin_tfidf)
                bin_centroids[time_bin] = np.asarray(bin_embeddings).mean(axis=0)

# TODO: compute cosine drift between consecutive bins
drift_rows = []
centroid_time_bins = [tb for tb in embedding_bins if str(tb) in bin_centroids]
for left_bin, right_bin in zip(centroid_time_bins, centroid_time_bins[1:]):
    cosine_drift = 1 - cosine_similarity(
        [bin_centroids[str(left_bin)]], [bin_centroids[str(right_bin)]]
    )[0, 0]
    drift_rows.append({
        "time_bin_from": str(left_bin),
        "time_bin_to": str(right_bin),
        "cosine_drift": float(cosine_drift),
        "representation": representation,
    })
drift_df = pd.DataFrame(
    drift_rows,
    columns=["time_bin_from", "time_bin_to", "cosine_drift", "representation"],
)

# TODO: plot embedding drift line chart and save to ARTIFACT_DIR / "embedding_drift_over_time.png"
transition_labels = drift_df["time_bin_from"] + " → " + drift_df["time_bin_to"]
fig, ax = plt.subplots(figsize=(max(8, len(drift_df) * 0.45 + 2), 4))
ax.plot(np.arange(len(drift_df)), drift_df["cosine_drift"], marker="o")
for idx in drift_df["cosine_drift"].nlargest(min(2, len(drift_df))).index:
    row = drift_df.loc[idx]
    ax.annotate(
        transition_labels.loc[idx],
        (idx, row["cosine_drift"]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8,
    )
ax.set_xticks(np.arange(len(drift_df)))
ax.set_xticklabels(transition_labels, rotation=45, ha="right")
ax.set_title(f"Temporal embedding centroid drift - {TARGET_CATEGORY}")
ax.set_ylabel("Cosine drift")
ax.set_xlabel("Adjacent time-bin transition")
ax.grid(alpha=0.2)
save_figure("embedding_drift_over_time.png")
plt.show()

# TODO: save drift_df to ARTIFACT_DIR / "embedding_centroid_drift.csv"
save_table(drift_df, "embedding_centroid_drift.csv")
if not drift_df.empty:
    largest_drift = drift_df.nlargest(1, "cosine_drift").iloc[0]
    print(
        f"Largest embedding drift: {largest_drift['time_bin_from']} -> "
        f"{largest_drift['time_bin_to']} ({largest_drift['cosine_drift']:.3f})"
    )

# Interpretation: The largest drift is 2007Q1 -> 2007Q2, with 2007Q2 -> 2007Q3 second.
# The TF-IDF table also changes across these early bins, but common dim-sum terms remain central.
# These spikes are close to the minimum-support boundary, so sampling change is more plausible than a causal cuisine shift.

# --- Validation ---
assert drift_df is not None, "drift_df must be computed"
assert (ARTIFACT_DIR / "embedding_centroid_drift.csv").exists()
print(f"\u2713 Embedding drift: {len(drift_df)} bin-transition records saved")




## Visual Evidence and Interpretation

Visualizations in this notebook are evidence, not decoration. Each figure should support interpretation, method comparison, diagnostic analysis, or communication of a discovered Yelp restaurant-intelligence pattern.

In a student report, explain what each selected figure reveals and one limitation of the visualization, especially when smoke-test data are small or sparse.

In [ ]:
# Visual evidence layer: compact temporal trends, heatmaps, and notable-period support.
import math
VIS_TOP_DISHES = min(8, len(rate_wide.index)) if "rate_wide" in globals() else 0

if "rate_wide" in globals() and isinstance(rate_wide, pd.DataFrame) and not rate_wide.empty and VIS_TOP_DISHES >= 1:
    totals = rate_wide.sum(axis=1, skipna=True).sort_values(ascending=False)
    top_dishes = totals.head(VIS_TOP_DISHES).index.tolist()
    ncols = 2 if len(top_dishes) > 1 else 1
    nrows = math.ceil(len(top_dishes) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, max(3, 2.4 * nrows)), squeeze=False, sharex=True)
    x = np.arange(len(rate_wide.columns))
    labels = rate_wide.columns.tolist()
    for ax, dish in zip(axes.ravel(), top_dishes):
        shade_unsupported_bins(ax, labels)
        ax.plot(x, rate_wide.loc[dish, labels], marker="o")
        ax.set_title(str(dish), fontsize=10)
        ax.set_ylabel("mentions / 1,000 reviews")
        ax.grid(alpha=0.2)
    for ax in axes.ravel()[len(top_dishes):]:
        ax.axis("off")
    for ax in axes[-1, :]:
        set_time_axis_ticks(ax, labels, max_ticks=8)
    fig.suptitle(f"Normalized mention trends for top dishes - {TARGET_CATEGORY}", y=1.02)
    fig.tight_layout()
    save_figure("task5_top_dish_trends_small_multiples.png")
    plt.show()
else:
    print("Skipping dish trend small multiples: not enough dish/time data.")



### Small multiples: reading dish trends as a comparative panel

The small-multiples grid plots each dish's normalized mention rate independently. This layout makes it easy to compare trajectory *shapes* across dishes, not just absolute levels. Key visual patterns:

- **Flat line at zero**: the dish has almost no mentions in your subset — it may not belong in a main-analysis report.
- **Rising staircase**: genuine popularity growth, or growing review coverage for that dish category.
- **Single spike**: a one-quarter anomaly, possibly from a viral review, a new restaurant opening, or a data artifact. Never trend-interpret a single spike.
- **Declining then flat**: the dish may have been a passing trend, or the business set changed.

> Cross-reference against `dish_evidence_tier_summary.csv` from Task 4. A dish rated `sparse` or `not_rankable` there is unlikely to show a stable trend here. Do not build a temporal argument on a dish with fewer than five supported time-bin observations.

In [ ]:
if "rate_wide" in globals() and isinstance(rate_wide, pd.DataFrame) and not rate_wide.empty:
    top_dishes = rate_wide.sum(axis=1, skipna=True).sort_values(ascending=False).head(max(1, VIS_TOP_DISHES)).index.tolist()
    heat = rate_wide.loc[top_dishes]
    save_table(heat.reset_index(), "task5_temporal_dish_heatmap_values.csv")
    fig, ax = plt.subplots(figsize=(max(8, len(heat.columns) * 0.45 + 3), max(3.5, len(heat.index) * 0.45 + 2)))
    masked_values = np.ma.masked_invalid(heat.values.astype(float))
    cmap = plt.cm.viridis.copy()
    cmap.set_bad("#eeeeee")
    im = ax.imshow(masked_values, aspect="auto", cmap=cmap)
    set_time_axis_ticks(ax, heat.columns.tolist(), max_ticks=16)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index)
    ax.set_title("Temporal dish mention heatmap")
    fig.colorbar(im, ax=ax, label="mention reviews per 1,000 reviews", fraction=0.046, pad=0.04)
    fig.tight_layout()
    save_figure("task5_temporal_dish_heatmap.png")
    plt.show()
else:
    print("Skipping temporal dish heatmap: rate_wide is unavailable or empty.")



### Temporal dish heatmap: patterns across all dishes simultaneously

The heatmap shows all top dishes × all time bins in one view. It is the most information-dense temporal visualization in this notebook.

**Gray cells** = unsupported bins (no data or below threshold). **Light cells** = low normalized mention rate. **Dark cells** = high normalized mention rate.

Things to look for:
1. **Diagonal bands** — if multiple dishes become more prominent in the same recent period, that may reflect overall platform growth rather than dish-specific trends.
2. **Row-by-row differences** — dishes with very different temporal patterns are the most analytically interesting: one rising while another falls in the same period suggests a menu or preference shift within the cuisine.
3. **Column-by-column consistency** — if every dish shows a spike in the same quarter, look for a volume anomaly in that quarter (check `review_counts_and_ratings_by_time.csv`).

In [ ]:
# TODO (Category D — Medium pipeline): Build notable-period annotation summary.
# Estimated time: ~1 h
#
# Create notable_summary_rows: a list of dicts describing the most important temporal events.
#
# For each notable observation:
#   {"notable_period_type": "...", "time_bin": "...", "value": ...}
#
# Include at minimum:
#   - Highest review volume bin
#   - Lowest review volume bin (from supported bins)
#   - Highest average sentiment bin (if sentiment_trends available)
#   - Lowest average sentiment bin
#   - Highest mention rate for the most-mentioned dish (from rate_wide)
#
# Save as task5_notable_period_annotations.csv.

notable_summary_rows = []
supported = temporal_summary[temporal_summary["meets_min_reviews"]].copy() if "temporal_summary" in globals() and not temporal_summary.empty else pd.DataFrame()

# TODO: add rows for highest/lowest review volume from supported bins
if not supported.empty:
    high_volume = supported.sort_values(["review_count", "time_bin"], ascending=[False, True]).iloc[0]
    low_volume = supported.sort_values(["review_count", "time_bin"], ascending=[True, True]).iloc[0]
    notable_summary_rows.extend([
        {
            "notable_period_type": "highest_review_volume",
            "time_bin": str(high_volume["time_bin"]),
            "value": float(high_volume["review_count"]),
        },
        {
            "notable_period_type": "lowest_supported_review_volume",
            "time_bin": str(low_volume["time_bin"]),
            "value": float(low_volume["review_count"]),
        },
    ])
# TODO: add rows for highest/lowest sentiment from sentiment_trends (if available)
if isinstance(sentiment_trends, pd.DataFrame) and not sentiment_trends.empty:
    supported_bins = set(supported["time_bin"].astype(str))
    sentiment_supported = sentiment_trends[
        sentiment_trends["time_bin"].astype(str).isin(supported_bins)
    ].dropna(subset=["avg_lexicon_sentiment"])
    if not sentiment_supported.empty:
        high_sentiment = sentiment_supported.sort_values(
            ["avg_lexicon_sentiment", "time_bin"], ascending=[False, True]
        ).iloc[0]
        low_sentiment = sentiment_supported.sort_values(
            ["avg_lexicon_sentiment", "time_bin"], ascending=[True, True]
        ).iloc[0]
        notable_summary_rows.extend([
            {
                "notable_period_type": "highest_average_lexicon_sentiment",
                "time_bin": str(high_sentiment["time_bin"]),
                "value": float(high_sentiment["avg_lexicon_sentiment"]),
            },
            {
                "notable_period_type": "lowest_average_lexicon_sentiment",
                "time_bin": str(low_sentiment["time_bin"]),
                "value": float(low_sentiment["avg_lexicon_sentiment"]),
            },
        ])
# TODO: add row for highest dish mention rate from rate_wide (if available)
if isinstance(rate_wide, pd.DataFrame) and not rate_wide.empty:
    most_mentioned_dish = rate_wide.sum(axis=1, skipna=True).idxmax()
    most_mentioned_series = rate_wide.loc[most_mentioned_dish].dropna()
    if not most_mentioned_series.empty:
        notable_summary_rows.append({
            "notable_period_type": "highest_mention_rate_for_most_mentioned_dish",
            "time_bin": str(most_mentioned_series.idxmax()),
            "value": float(most_mentioned_series.max()),
        })

notable_summary_df = pd.DataFrame(notable_summary_rows) if notable_summary_rows else pd.DataFrame(columns=["notable_period_type","time_bin","value"])
# save_table(notable_summary_df, "task5_notable_period_annotations.csv")
save_table(notable_summary_df, "task5_notable_period_annotations.csv")
# display(notable_summary_df)
display(notable_summary_df)

# --- Validation ---
print(f"\u2713 Notable annotations: {len(notable_summary_df)} entries")


## Temporal Evidence Summary

This section exports the core temporal analysis results for interpretation and downstream reuse.

### Dish Trend Direction and Reliability Labels

This summary keeps trend direction separate from evidence reliability. It does not combine different signals into a weighted numerical score.

Direction uses the normalized early/late mention-rate change and the Spearman trend. A dish is labeled `rising` or `declining` when both directions agree and the exploratory unadjusted p-value is below 0.05. A p-value from 0.05 to 0.10 produces a `possible` label. Larger p-values produce `no clear trend`, while opposing signs produce `mixed direction`.

Reliability uses transparent diagnostic rules. `High` requires a clear direction, dish support in at least half of the supported bins, at least five supported dish bins in both periods, stable direction after removing the first and last supported bins, and no dominant calendar-quarter risk. `Medium` requires a clear or possible direction, at least five supported dish bins overall, at least two in both periods, and boundary stability. All other cases are `Low`. The seasonality flag is raised when the range across calendar-quarter means is larger than the early/late change. Sentiment agreement is reported as context only and does not determine popularity direction.

In [ ]:
# Xiaofei
# Build categorical dish-trend direction and reliability labels without a weighted score.
from scipy.stats import spearmanr

DIRECTION_P_CLEAR = 0.05
DIRECTION_P_POSSIBLE = 0.10
MIN_BASIC_SUPPORTED_BINS = 5
MIN_BASIC_PERIOD_BINS = 2
MIN_HIGH_PERIOD_BINS = 5

full_label_axis = [str(tb) for tb in complete_time_order]
supported_label_axis = [str(tb) for tb in time_order]
full_axis_midpoint = max(1, len(full_label_axis) // 2)
label_early_bins = full_label_axis[:full_axis_midpoint]
label_late_bins = full_label_axis[full_axis_midpoint:]
supported_axis_midpoint = max(1, len(supported_label_axis) // 2)
sentiment_early_bins = supported_label_axis[:supported_axis_midpoint]
sentiment_late_bins = supported_label_axis[supported_axis_midpoint:]
trimmed_supported_axis = supported_label_axis[1:-1]
min_high_supported_bins = max(1, int(np.ceil(len(supported_label_axis) / 2)))

def safe_spearman(values):
    series = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if len(series) < 3 or series.nunique() < 2:
        return np.nan, np.nan
    rho, p_value = spearmanr(np.arange(len(series)), series.to_numpy())
    return float(rho), float(p_value)

def same_nonzero_sign(left, right):
    if pd.isna(left) or pd.isna(right) or left == 0 or right == 0:
        return False
    return bool(np.sign(left) == np.sign(right))

def direction_label(delta, rho, p_value):
    if pd.isna(delta) or pd.isna(rho) or pd.isna(p_value):
        return "insufficient evidence"
    if not same_nonzero_sign(delta, rho):
        return "mixed direction"
    direction = "rising" if delta > 0 else "declining"
    if p_value < DIRECTION_P_CLEAR:
        return direction
    if p_value < DIRECTION_P_POSSIBLE:
        return f"possible {direction}"
    return "no clear trend"

label_rows = []
dish_counts_for_labels = dish_counts.copy()
dish_counts_for_labels["time_bin"] = dish_counts_for_labels["time_bin"].astype(str)
sentiment_for_labels = dish_sentiment.copy() if isinstance(dish_sentiment, pd.DataFrame) else pd.DataFrame()
if not sentiment_for_labels.empty:
    sentiment_for_labels["time_bin"] = sentiment_for_labels["time_bin"].astype(str)

for dish in dish_df["dish"].astype(str):
    dish_data = dish_counts_for_labels[dish_counts_for_labels["dish"] == dish].copy()
    dish_data = dish_data.set_index("time_bin").reindex(full_label_axis)
    mention_rate = pd.to_numeric(dish_data["mention_rate"], errors="coerce")
    mention_reviews = pd.to_numeric(dish_data["mention_reviews"], errors="coerce")

    early_avg_rate = mention_rate.reindex(label_early_bins).mean()
    late_avg_rate = mention_rate.reindex(label_late_bins).mean()
    late_minus_early = late_avg_rate - early_avg_rate
    supported_rates = mention_rate.reindex(supported_label_axis).dropna()
    spearman_rho, spearman_p = safe_spearman(supported_rates)
    assigned_direction = direction_label(late_minus_early, spearman_rho, spearman_p)
    direction_consistent = same_nonzero_sign(late_minus_early, spearman_rho)

    supported_cell_mask = mention_reviews >= MIN_DISH_MENTION_REVIEWS_PER_BIN
    mention_supported_bins = int(supported_cell_mask.reindex(supported_label_axis).fillna(False).sum())
    early_supported_bins = int(supported_cell_mask.reindex(label_early_bins).fillna(False).sum())
    late_supported_bins = int(supported_cell_mask.reindex(label_late_bins).fillna(False).sum())
    total_mention_reviews = int(mention_reviews.reindex(supported_label_axis).fillna(0).sum())

    trimmed_rates = mention_rate.reindex(trimmed_supported_axis).dropna()
    trimmed_rho, _ = safe_spearman(trimmed_rates)
    trimmed_early = mention_rate.reindex([tb for tb in label_early_bins if tb in trimmed_supported_axis]).mean()
    trimmed_late = mention_rate.reindex([tb for tb in label_late_bins if tb in trimmed_supported_axis]).mean()
    trimmed_delta = trimmed_late - trimmed_early
    boundary_stable = (
        same_nonzero_sign(late_minus_early, trimmed_delta)
        and same_nonzero_sign(spearman_rho, trimmed_rho)
    )

    seasonality_range = np.nan
    dominant_calendar_quarter = pd.NA
    seasonality_risk = False
    if TIME_GRANULARITY == "quarter" and not supported_rates.empty:
        seasonal_frame = supported_rates.rename("mention_rate").reset_index()
        seasonal_frame.columns = ["time_bin", "mention_rate"]
        seasonal_frame["calendar_quarter"] = seasonal_frame["time_bin"].str.extract(r"Q([1-4])", expand=False)
        quarter_means = seasonal_frame.dropna(subset=["calendar_quarter"]).groupby("calendar_quarter")["mention_rate"].mean()
        if len(quarter_means) >= 3:
            seasonality_range = float(quarter_means.max() - quarter_means.min())
            dominant_calendar_quarter = f"Q{quarter_means.idxmax()}"
            seasonality_risk = bool(seasonality_range > abs(late_minus_early))

    dish_sentiment_rows = sentiment_for_labels[sentiment_for_labels["dish"] == dish] if not sentiment_for_labels.empty else pd.DataFrame()
    early_sentiment_rows = dish_sentiment_rows[dish_sentiment_rows["time_bin"].isin(sentiment_early_bins)] if not dish_sentiment_rows.empty else pd.DataFrame()
    late_sentiment_rows = dish_sentiment_rows[dish_sentiment_rows["time_bin"].isin(sentiment_late_bins)] if not dish_sentiment_rows.empty else pd.DataFrame()
    early_sentiment = early_sentiment_rows["avg_lexicon_sentiment"].mean() if not early_sentiment_rows.empty else np.nan
    late_sentiment = late_sentiment_rows["avg_lexicon_sentiment"].mean() if not late_sentiment_rows.empty else np.nan
    sentiment_delta = late_sentiment - early_sentiment if pd.notna(early_sentiment) and pd.notna(late_sentiment) else np.nan
    if pd.isna(sentiment_delta) or sentiment_delta == 0 or late_minus_early == 0:
        sentiment_relation = "unavailable or flat"
    elif same_nonzero_sign(late_minus_early, sentiment_delta):
        sentiment_relation = "same direction"
    else:
        sentiment_relation = "opposite direction"

    strong_support = (
        mention_supported_bins >= min_high_supported_bins
        and early_supported_bins >= MIN_HIGH_PERIOD_BINS
        and late_supported_bins >= MIN_HIGH_PERIOD_BINS
    )
    basic_support = (
        mention_supported_bins >= MIN_BASIC_SUPPORTED_BINS
        and early_supported_bins >= MIN_BASIC_PERIOD_BINS
        and late_supported_bins >= MIN_BASIC_PERIOD_BINS
    )
    clear_direction = assigned_direction in {"rising", "declining"}
    possible_direction = assigned_direction in {"possible rising", "possible declining"}
    if clear_direction and strong_support and boundary_stable and not seasonality_risk:
        reliability_label = "High"
    elif (clear_direction or possible_direction) and basic_support and boundary_stable:
        reliability_label = "Medium"
    else:
        reliability_label = "Low"

    reliability_basis = "; ".join([
        f"{mention_supported_bins}/{len(supported_label_axis)} dish-supported bins",
        f"early/late support {early_supported_bins}/{late_supported_bins}",
        "direction signals agree" if direction_consistent else "direction signals conflict",
        "boundary stable" if boundary_stable else "boundary sensitive",
        "seasonality risk" if seasonality_risk else "no dominant seasonality risk",
    ])

    label_rows.append({
        "dish": dish,
        "early_avg_mention_rate": early_avg_rate,
        "late_avg_mention_rate": late_avg_rate,
        "late_minus_early": late_minus_early,
        "spearman_rho": spearman_rho,
        "spearman_p_unadjusted": spearman_p,
        "direction_label": assigned_direction,
        "mention_supported_bins": mention_supported_bins,
        "early_supported_bins": early_supported_bins,
        "late_supported_bins": late_supported_bins,
        "total_mention_reviews": total_mention_reviews,
        "boundary_direction_stable": boundary_stable,
        "seasonality_range_per_1000": seasonality_range,
        "dominant_calendar_quarter": dominant_calendar_quarter,
        "seasonality_risk": seasonality_risk,
        "early_sentiment_cells": len(early_sentiment_rows),
        "late_sentiment_cells": len(late_sentiment_rows),
        "sentiment_delta": sentiment_delta,
        "sentiment_relation_to_mentions": sentiment_relation,
        "reliability_label": reliability_label,
        "reliability_basis": reliability_basis,
    })

dish_trend_direction_reliability = pd.DataFrame(label_rows)
reliability_order = {"High": 0, "Medium": 1, "Low": 2}
dish_trend_direction_reliability["_reliability_order"] = dish_trend_direction_reliability["reliability_label"].map(reliability_order)
dish_trend_direction_reliability = dish_trend_direction_reliability.sort_values(
    ["_reliability_order", "late_minus_early"], ascending=[True, False]
).drop(columns="_reliability_order").reset_index(drop=True)
save_table(dish_trend_direction_reliability, "dish_trend_direction_reliability_labels.csv")
display(dish_trend_direction_reliability[[
    "dish", "direction_label", "reliability_label",
    "late_minus_early", "spearman_rho", "mention_supported_bins",
    "boundary_direction_stable", "seasonality_risk",
    "sentiment_relation_to_mentions",
]])

assert len(dish_trend_direction_reliability) == len(dish_df)
assert set(dish_trend_direction_reliability["reliability_label"]).issubset({"High", "Medium", "Low"})
assert (ARTIFACT_DIR / "dish_trend_direction_reliability_labels.csv").exists()
print(f"✓ Direction/reliability labels saved for {len(dish_trend_direction_reliability)} dishes")


In [ ]:
# TODO (Category D — Medium pipeline): Save all required temporal evidence exports.
# Estimated time: ~0.5 h
#
# Required exports for the report and Task 6 temporal feature pipeline:
#   temporal_review_subset.csv         — the full review subset with time_bins
#   time_axis_support.csv              — supported vs. sparse bins
#   review_counts_and_ratings_by_time.csv — per-bin review count and avg_stars
#   dish_mentions_by_time_long.csv     — long-form dish mention counts by bin
#   dish_sentiment_by_time.csv         — dish-level sentiment by time bin
#   temporal_sentiment_trends.csv      — aggregate sentiment per time bin
#   top_terms_by_time.csv              — top TF-IDF terms per bin
#   task5_notable_period_annotations.csv — notable-period summary
#
# For each variable, check if it exists and is non-empty before saving.
# Use save_table(df, filename) or df.to_csv(ARTIFACT_DIR / filename).
# Print "Saved: <filename>" for each successful save.

# TODO: save subset_df if available
def save_if_nonempty_df(var_name: str, filename: str) -> None:
    df = globals().get(var_name)
    if isinstance(df, pd.DataFrame) and not df.empty:
        save_table(df, filename)
        print(f"Saved: {filename}")
    else:
        print(f"Skipped empty or missing DataFrame: {var_name} -> {filename}")

save_if_nonempty_df("subset_df", "temporal_review_subset.csv")
# TODO: save time_axis_support if available
save_if_nonempty_df("time_axis_support", "time_axis_support.csv")
# TODO: save review_counts_and_ratings if available
save_if_nonempty_df("review_counts_and_ratings", "review_counts_and_ratings_by_time.csv")
# TODO: save dish_counts with appropriate column renaming as dish_mentions_by_time_long.csv
dish_counts_to_save = globals().get("dish_counts")
if isinstance(dish_counts_to_save, pd.DataFrame) and not dish_counts_to_save.empty:
    save_table(
        dish_counts_to_save.rename(
            columns={"mention_rate": "mention_reviews_per_1000_reviews"}
        ),
        "dish_mentions_by_time_long.csv",
    )
    print("Saved: dish_mentions_by_time_long.csv")
else:
    print("Skipped empty or missing DataFrame: dish_counts -> dish_mentions_by_time_long.csv")
# TODO: save dish_sentiment as dish_sentiment_by_time.csv
save_if_nonempty_df("dish_sentiment", "dish_sentiment_by_time.csv")
# TODO: save sentiment_trends as temporal_sentiment_trends.csv
save_if_nonempty_df("sentiment_trends", "temporal_sentiment_trends.csv")
# TODO: save top_terms_df as top_terms_by_time.csv
save_if_nonempty_df("top_terms_df", "top_terms_by_time.csv")
# TODO: save notable_summary_df as task5_notable_period_annotations.csv
save_if_nonempty_df("notable_summary_df", "task5_notable_period_annotations.csv")
# Save the additional categorical direction/reliability summary.
save_if_nonempty_df("dish_trend_direction_reliability", "dish_trend_direction_reliability_labels.csv")

# --- Validation ---
core_files = [
    "temporal_review_subset.csv", "time_axis_support.csv",
    "review_counts_and_ratings_by_time.csv", "dish_mentions_by_time_long.csv",
    "dish_sentiment_by_time.csv", "temporal_sentiment_trends.csv",
    "top_terms_by_time.csv", "task5_notable_period_annotations.csv",
]
missing = [f for f in core_files if not (ARTIFACT_DIR / f).exists()]
print("Missing artifacts:", missing if missing else "None — all saved!")


### Temporal exports: what each table captures

`temporal_review_subset.csv` — the review records used, with time bins attached.

`dish_mentions_by_time_long.csv` — one row per (dish, time_bin) with mention count.

`dish_sentiment_by_time.csv` — one row per (dish, time_bin) with dish-level sentiment.

`temporal_sentiment_trends.csv` — overall sentiment scores by time bin.

`top_terms_by_time.csv` — most distinctive terms per time period.

`dish_trend_direction_reliability_labels.csv` — categorical direction and reliability evidence for each tracked dish.

## 11. Required comparison and interpretation

A strong Task 5 report should answer the following questions:

- **Temporal coverage**: what time periods does your analysis cover? Are they evenly supported?
- **Trend vs. noise**: do observed trends appear real, or could they reflect seasonal effects
  or sparse data?
- **Dish popularity changes**: which dishes rise or fall in mention frequency over time?
  What might explain those changes?
- **Sentiment evolution**: does sentiment around specific dishes change over time?
  Do sentiment and rating trends agree?
- **Distinctive terms**: which terms become more prominent in specific time periods?
  What do they suggest about changes in the cuisine or the platform?

**Insights! 💡**

Task 5’s temporal mining sits between classical time-series analysis and text mining.

**Classical time-series era:** moving averages and smoothing helped analysts separate signal from noise in regular numeric sequences.

**Text-mining era:** TF-IDF by time slice and temporal topic analysis made it possible to ask which words and themes are distinctive in each period.

**Modern review-mining perspective:** Yelp data is irregular, sparse, and behaviorally messy. The core skill is not fitting the fanciest temporal model; it is making defensible claims by combining support counts, normalized trends, sentiment/rating movement, distinctive terms, and example reviews.


## 12. Deliverables checklist

For the Task 5 report, you should include:

- one temporal EDA visualisation (volume or rating trend);
- evidence for at least one temporal pattern (rising/falling dish, sentiment shift);
- a table or figure showing top dishes or terms in contrasting time periods;
- interpretation of whether changes are real, seasonal, or artefacts of sparse data.

In [ ]:
core_artifacts = [
    "temporal_review_subset.csv",
    "time_axis_support.csv",
    "review_counts_and_ratings_by_time.csv",
    "dish_mentions_by_time_long.csv",
    "dish_sentiment_by_time.csv",
    "temporal_sentiment_trends.csv",
    "top_terms_by_time.csv",
    "task5_notable_period_annotations.csv",
]
missing = [f for f in core_artifacts if not (ARTIFACT_DIR / f).exists()]
if missing:
    print("⚠  Missing artifacts:", missing)
else:
    print("✓ All required artifacts present.")


### What the artifact status check tells you

The table above lists every expected output file and whether it exists. A `False` in the `exists` column means a section did not run to completion or raised an exception that was silently caught.

**Common causes of missing artifacts:**
- `time_bin_support_all.csv` / `time_axis_support.csv` missing → Section 3 (temporal subset construction) did not complete. Check if the temporal subset DataFrame was empty.
- `dish_mentions_by_time_long.csv` missing → Section 6 (dish mention counting) raised an error, probably because no dish matched any review. Check the dish list and the category target.
- Optional visual artifacts (`task5_top_dish_trends_small_multiples.png`, etc.) missing → these are generated only when there are enough supported dish-time cells. A warning (not an error) is raised for these.

> **If you see missing required artifacts:** rerun the notebook **top-to-bottom** in a clean kernel. Do not try to selectively rerun individual cells after a partial failure — cell-level rerunning in a broken state creates dependency inconsistencies.

## Final artifact validation

Run this cell before submission. It checks that required artifacts exist, are non-empty, and satisfy the most important schema expectations. Optional artifacts are reported but do not block submission.

In [ ]:
try:
    temporal_check = pd.read_csv(ARTIFACT_DIR / "temporal_review_subset.csv")
    print(f"Temporal review subset: {len(temporal_check)} rows")
    time_bins = temporal_check["time_bin"].nunique() if "time_bin" in temporal_check.columns else "N/A"
    print(f"Time bins covered: {time_bins}")
except Exception as e:
    print(f"Could not read temporal_review_subset.csv: {e}")

artifact_status = pd.DataFrame([
    {"artifact": f, "exists": (ARTIFACT_DIR / f).exists()}
    for f in core_artifacts
])
display(artifact_status)

## Submission reminders

Before submission, verify that required Task 5 artifacts exist under
`artifacts/task5` and that sparse time bins are clearly marked. Do not
treat visible trends as forecasts or causal claims. Your interpretation
should discuss data support, normalization, seasonality, platform activity,
and sampling artifacts.
